# 欠測下の巡回構造: 既存手法の検証コード

卒業研究「欠測下で巡回構造を許す制約ベース因果探索アルゴリズムの提案」のうち、既存手法の検証に使ったコードを1つのノートブックにまとめたもの (提案手法は含まない)。
巡回構造と欠測機構を別々に制御できる合成データを作り、因果探索の既存手法が破綻する条件を調べる。
各モジュールをこのノートブックの中で定義している (名前の衝突を避けるため、モジュールごとに名前空間を分けている)。
docstring の「実行: uv run python src/名前.py ...」は .py に分かれていた頃の書き方で、このノートブックでは run_main("名前", ...) に当たる。
src/r/run_mvpc_official.R と src/runners/missnodag_runner.py は、下の %%writefile のセルが書き出す。
ノートブックの中の「図1」と「中間発表」は、対象とする構造を描いた中間発表 (2026-08-06) の図と資料を指す。構造は 2.2 の generator の docstring にある。

## 構成

1. 準備 (設定・モジュール定義の仕組み・外部ランナーの書き出し)
2. 共通部品: 評価指標, 生成器, d分離, 既存手法のラッパ, 公式実装のラッパ, 診断, 実験ランナー
3. 9月の検証 (識別可能性, 欠測が条件付き独立を壊す条件, 偽の巡回, 4軸の比較)
4. 10月の検証 (公式実装との突き合わせ, 評価の正解の置き方, 公式実装での偽の巡回, MVPC の移植の差)

## 使い方

- リポジトリの直下をカレントディレクトリにして開く。公式実装は先に scripts/setup_official.sh で official/ に入れておく (README.md)。
- QUICK = True (既定) では、反復数と標本数を小さくして全体を短時間で動かす。結果は outputs_quick/ と figures_quick/ に書く。
- QUICK = False にすると検証と同じ設定で回り、結果は outputs/ と figures/ に書く。全部で数十時間かかる
  (公式実装を入れた4軸の実験だけで約26時間)。RUN で実験ごとに回すかを選べる。

## 公式実装の前提

公式実装は official/ の下に取得したものを呼ぶ。中身には手を入れていない。

- Tetrad (CCD・FCI・LiNG-D): official/py-tetrad の jar と official/jdk の Java 21
- MVPC: official/MVPC (R)。R パッケージは official/Rlib
- MissDAG: official/MissDAG
- MissNODAG: official/missnodag。依存の版が固定されているので official/missnodag/.venv の Python で別に動かす

## 1. 準備

In [ ]:
import contextlib
import json
import linecache
import os
import sys
import types

from IPython.core.magic import register_cell_magic

# このノートブックはリポジトリの直下で開く
PROJECT = os.path.abspath(os.getcwd())
assert os.path.isdir(os.path.join(PROJECT, "official")), "リポジトリの直下で開くこと (official/ が要る)"
SRC_DIR = os.path.join(PROJECT, "src")
# 外部の処理系で動かすファイル (R, MissNODAG のランナー) の書き出し先
os.makedirs(os.path.join(SRC_DIR, "r"), exist_ok=True)
os.makedirs(os.path.join(SRC_DIR, "runners"), exist_ok=True)

QUICK = True     # False で検証と同じ設定 (数十時間)

RUN = {k: True for k in [
    "diagnostics", "dsep", "attenuation", "ci_audit", "cycle_missing_probe",
    "cycle_identification_break", "phantom_cycle", "run_experiment", "make_figures",
    "env_report", "compare_official", "eval_oracle", "phantom_official", "mvpc_seed_check",
    "mvpc_bug_effect", "mvpc_port_diff", "phantom_nonmonotone", "missnodag_axes",
    "run_experiment_official", "run_experiment_clfix"]}

OUT_DIR = os.path.join(PROJECT, "outputs_quick" if QUICK else "outputs")
FIG_DIR = os.path.join(PROJECT, "figures_quick" if QUICK else "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print("PROJECT:", PROJECT)
print("QUICK:", QUICK, "->", OUT_DIR)

%%module 名前 で始まるセルは、そのセルの中身を1つのモジュールとして定義する。
他のモジュールからは普通の import (例: from generator import generate) で使える。
モジュールの __file__ は src/名前.py に置くので、モジュール内の相対パス (outputs/ や official/) は元のまま解決される。
QUICK のときは出力先 (OUT, FIG) だけを outputs_quick/ と figures_quick/ に差し替える。

In [ ]:
@register_cell_magic
def module(line, cell):
    name = line.strip()
    mod = types.ModuleType(name)
    mod.__file__ = os.path.join(SRC_DIR, name + ".py")
    sys.modules[name] = mod
    # トレースバックにノートブックのセルの中身が出るよう, 別名で linecache に登録する
    fname = f"<module {name}>"
    linecache.cache[fname] = (len(cell), None, cell.splitlines(True), fname)
    exec(compile(cell, fname, "exec"), mod.__dict__)
    if hasattr(mod, "OUT"):
        mod.OUT = OUT_DIR
    if hasattr(mod, "FIG"):
        mod.FIG = FIG_DIR


def run_main(name, *args):
    """コマンドラインで python src/name.py args... と打ったのと同じことをする.

    run_experiment は methods.METHODS を書き換えるので, 呼ぶたびに元に戻す.
    """
    mod = sys.modules[name]
    meths = sys.modules.get("methods")
    saved = dict(meths.METHODS) if meths else None
    old = sys.argv
    sys.argv = [name + ".py", *map(str, args)]
    try:
        mod.main()
    finally:
        sys.argv = old
        if meths is not None:
            meths.METHODS.clear()
            meths.METHODS.update(saved)


### 外部の処理系で動かすファイル

MVPC の公式実装は R、MissNODAG は依存の版が固定された別の Python 環境で動くので、
このノートブックからは呼び出せない。次の2つのセルでファイルとして書き出し、ラッパから subprocess で呼ぶ。

In [ ]:
%%writefile src/r/run_mvpc_official.R
# 著者公式実装 (https://github.com/TURuibo/MVPC, official/MVPC) で MVPC を回す.
# 使い方: Rscript run_mvpc_official.R <入力CSV> <出力JSON> <alpha> <method>
#   method: td (検定ごと削除の PC) / permc (MVPC + PermC) / drw (MVPC + DRW)
# 出力: amat[i, j] = 1 は i -> j の辺 (無向辺は両方 1). pcalg の graphNEL を行列にしたもの.
args <- commandArgs(trailingOnly = TRUE)
here <- normalizePath(dirname(sub("--file=", "", grep("--file=", commandArgs(FALSE), value = TRUE))))
root <- normalizePath(file.path(here, "..", ".."))
.libPaths(c(file.path(root, "official", "Rlib"), .libPaths()))
suppressPackageStartupMessages({
  src <- file.path(root, "official", "MVPC", "src")
  source(file.path(src, "CITest.R"))
  source(file.path(src, "MissingValuePC.R"))
  library(jsonlite)
})
X <- as.matrix(read.csv(args[1], check.names = FALSE))
alpha <- as.numeric(args[3]); method <- args[4]
p <- ncol(X); labels <- colnames(X)
set.seed(if (length(args) >= 5) as.integer(args[5]) else 0)
t0 <- Sys.time()
res <- switch(method,
  td    = pc(list(data = X), gaussCItest.td, alpha = alpha, labels = labels),
  permc = mvpc(list(data = X), gaussCItest.td, gaussCItest.permc, alpha = alpha, p = p),
  drw   = mvpc(list(data = X), gaussCItest.td, gaussCItest.drw, alpha = alpha, p = p))
amat <- as(res@graph, "matrix")
prt <- if (method == "td") NULL else tryCatch(detection_prt_m(X, gaussCItest.td, alpha, p), error = function(e) NULL)
write_json(list(amat = unname(amat), labels = labels, method = method,
                seconds = as.numeric(difftime(Sys.time(), t0, units = "secs")),
                prt_m = prt), args[2], auto_unbox = TRUE, digits = NA)

In [ ]:
%%writefile src/runners/missnodag_runner.py
"""著者公式実装 (https://github.com/muralikgs/missnodag, official/missnodag) で MissNODAG を回す.

MissNODAG は依存の版を固定している (numpy<=1.26, scipy 1.9.3, scikit-learn 1.1.3 など) ので,
本体の .venv ではなく official/missnodag/.venv (uv sync で作る) の Python で実行する.
src/official_missing.py から subprocess で呼ばれる.

使い方: python missnodag_runner.py <入力 .npz> <出力 .json> [max_epochs] [seed]
  入力 X: (n, d), 欠測は NaN. 観測データのみ (介入なし) として渡す.
ハイパーパラメータは論文の実験設定 (benchmarks/generate_settings/ablation_cycles.py) のもの.
学習は benchmarks/run_benchmark_per_setting.py の train_missnodag と同じ手順で行う.

data_missing=True のとき Trainer は X_clean を使わないが, 漏れが起きえないように
X_clean には欠測を 0 で埋めた行列を渡す.
"""
import json
import os
import sys
import time

REPO = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..", "official", "missnodag"))
sys.path.insert(0, REPO)

import numpy as np  # noqa: E402
import torch  # noqa: E402

from datagen.missingData import missing_value_dataset  # noqa: E402
from models.nodags.functions import gumbelSoftMLP  # noqa: E402
from models.nodags.missing_data_model import missModel  # noqa: E402
from models.nodags.missingness_mechanism import identifiableMNAR  # noqa: E402
from models.nodags.resblock import iResBlock  # noqa: E402
from models.nodags.trainer import Trainer  # noqa: E402

MODEL_PARAMS = {  # ablation_cycles.py の model_params
    "activation": "tanh", "max_epochs": 100, "batch_size": 512, "lr": 1e-1, "lc": 1e-3,
    "missing-mech-sparsity-reg": 1e-2, "ldag": 5e-1, "lnc": 1e-1,
    "adj_threshold": 0.7, "mm_threshold": 0.2,
}


def main():
    inp, out = sys.argv[1], sys.argv[2]
    cfg = dict(MODEL_PARAMS)
    if len(sys.argv) > 3:
        cfg["max_epochs"] = int(sys.argv[3])
    seed = int(sys.argv[4]) if len(sys.argv) > 4 else 0
    np.random.seed(seed)
    torch.manual_seed(seed)

    X = np.load(inp)["X"].astype(float)
    n, d = X.shape
    R = (~np.isnan(X)).astype(float)
    X_fill = np.nan_to_num(X, nan=0.0)
    data = missing_value_dataset([(X_fill, R, X)], [[None]])

    causal_mech = gumbelSoftMLP(n_nodes=d, lip_constant=0.9, activation=cfg["activation"])
    nodags = iResBlock(func=causal_mech, n_power_series=None)
    missing_mech = identifiableMNAR(n_nodes=d, learning_method="em",
                                    constraint_optimization="aug-lagrangian",
                                    C=cfg["missing-mech-sparsity-reg"])
    miss_model = missModel(nodags, missing_mech=missing_mech, is_mcar=False)
    trainer = Trainer(miss_model=miss_model, lr=cfg["lr"], lr_miss=cfg["lr"],
                      lambda_c=cfg["lc"], max_epochs=cfg["max_epochs"],
                      batch_size=cfg["batch_size"], missingness_learn="em",
                      lambda_dag=cfg["ldag"], lambda_nc=cfg["lnc"], obs=False)
    t0 = time.time()
    err = None
    try:
        trainer.train(data=data, print_loss=False, print_interval=100, data_missing=True,
                      min_accept_factor=0.5, store_grad_norm=False)
    except Exception as e:  # noqa: BLE001
        err = f"{type(e).__name__}: {e}"
    W = miss_model.gen_model.get_w_adj()   # W[i, j] は i -> j の辺の確率
    json.dump({"w_adj": np.asarray(W).tolist(),
               "x2r": miss_model.missing_mech.coefs_x2r.detach().cpu().numpy().tolist(),
               "adj_threshold": cfg["adj_threshold"], "max_epochs": cfg["max_epochs"],
               "seconds": time.time() - t0, "error": err}, open(out, "w"))


if __name__ == "__main__":
    main()

## 2. 共通部品

### 2.1 評価指標 (metrics)

手法ごとに出力の形 (CPDAG, PAG, 重み行列) が違うので、共通の表現に変換してから測る。
from_causal_learn は 2026-10-11 に向きの誤りを直した。

In [ ]:
%%module metrics
"""推定結果を手法間で横並びに比較するための表現と指標.

手法ごとに出力の型が違う (PC は CPDAG, FCI は PAG, LiNGAM 系は重み行列) ので,
共通表現に落としてから測る. 共通表現は
    adj[i, j] = True   i -> j の矢が推定されている
    undirected(i, j)   向きが定まらない辺 (CPDAG の無向辺, PAG の丸印) がある
とし, 骨格 (skeleton) の指標を第一, 向きの指標を第二に置く.

骨格を第一に置く理由は, 巡回と欠測の下では向きの意味が手法ごとに違う一方,
「辺があるか」は全手法で共通に定義できるためである. 0618 の検証で
SHD が閾値と出力形式に強く依存することを確認しているので, SHD は補助に留め,
閾値に依存しない指標 (F1, 2-cycle の再現) を主に使う.
"""
from __future__ import annotations

import numpy as np


class Estimate:
    """推定された構造の共通表現.

    directed[i, j] : i -> j の矢があると推定された
    adjacent[i, j] : i と j の間に何らかの辺があると推定された (対称)
    bidirected[i,j]: i <-> j (潜在交絡) と推定された (対称)
    """

    def __init__(self, names: list[str], directed: np.ndarray,
                 adjacent: np.ndarray, bidirected: np.ndarray | None = None,
                 kind: str = "", raw=None):
        d = len(names)
        self.names = names
        self.directed = directed.astype(bool)
        self.adjacent = adjacent.astype(bool)
        self.bidirected = (np.zeros((d, d), bool) if bidirected is None
                           else bidirected.astype(bool))
        self.kind = kind
        self.raw = raw

    @property
    def undirected(self) -> np.ndarray:
        """辺はあるが向きが定まっていない対."""
        return self.adjacent & ~(self.directed | self.directed.T)


def true_structure(B_obs: np.ndarray, names: list[str]) -> Estimate:
    """係数行列 (B[i, j] は x_j -> x_i) から真の構造を共通表現にする."""
    directed = (np.abs(B_obs) > 0).T          # directed[i, j] = i -> j
    adjacent = directed | directed.T
    return Estimate(names, directed, adjacent, kind="true")


def from_causal_learn(G, names: list[str], kind: str) -> Estimate:
    """causal-learn の graph 行列を共通表現にする.

    causal-learn の符号則:
      G[j, i] = 1  かつ G[i, j] = -1   ->  i -> j
      G[i, j] = G[j, i] = -1           ->  i -- j
      G[i, j] = G[j, i] = 1            ->  i <-> j
      2 は PAG の丸印 (o)
    """
    d = len(names)
    G = np.asarray(G)
    directed = np.zeros((d, d), bool)
    adjacent = np.zeros((d, d), bool)
    bidirected = np.zeros((d, d), bool)
    for i in range(d):
        for j in range(i + 1, d):
            a, b = G[i, j], G[j, i]
            if a == 0 and b == 0:
                continue
            adjacent[i, j] = adjacent[j, i] = True
            # G[i, j] は i の側の印. i が尾 (-1) で j が矢 (1) なら i -> j.
            # 2026-10-11 修正: 9月版は a == 1 and b == -1 を i -> j としており,
            # causal-learn の出力の向きをすべて逆に読んでいた
            if a == -1 and b == 1:
                directed[i, j] = True
            elif a == 1 and b == -1:
                directed[j, i] = True
            elif a == 1 and b == 1:
                bidirected[i, j] = bidirected[j, i] = True
    return Estimate(names, directed, adjacent, bidirected, kind=kind, raw=G)


def from_weight_matrix(W: np.ndarray, names: list[str], thresh: float,
                       kind: str) -> Estimate:
    """LiNGAM 系の重み行列 (W[i, j] は x_j -> x_i) を共通表現にする."""
    A = (np.abs(np.nan_to_num(W)) > thresh).T
    np.fill_diagonal(A, False)
    return Estimate(names, A, A | A.T, kind=kind, raw=W)


# ---------------------------------------------------------------------------
# 指標
# ---------------------------------------------------------------------------
def _pairs(mask: np.ndarray) -> set[tuple[int, int]]:
    d = mask.shape[0]
    return {(i, j) for i in range(d) for j in range(i + 1, d) if mask[i, j] or mask[j, i]}


def skeleton_scores(est: Estimate, truth: Estimate) -> dict:
    t = _pairs(truth.adjacent)
    e = _pairs(est.adjacent)
    tp, fp, fn = len(t & e), len(e - t), len(t - e)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"skeleton_tp": tp, "skeleton_fp": fp, "skeleton_fn": fn,
            "skeleton_precision": prec, "skeleton_recall": rec,
            "skeleton_f1": f1, "skeleton_shd": fp + fn}


def orientation_scores(est: Estimate, truth: Estimate) -> dict:
    """骨格が当たっている辺のうち, 向きが正しく定まったものの割合.

    correct   : 真に i -> j のみで, 推定も i -> j のみ
    reversed  : 向きが逆
    undecided : 辺はあるが向きが定まっていない (無向・丸印・双方向)
    """
    correct = reversed_ = undecided = total = 0
    d = len(truth.names)
    for i in range(d):
        for j in range(i + 1, d):
            if not (truth.adjacent[i, j] and est.adjacent[i, j]):
                continue
            t_ij, t_ji = truth.directed[i, j], truth.directed[j, i]
            if t_ij and t_ji:
                continue   # 2-cycle は別指標で測る
            total += 1
            e_ij, e_ji = est.directed[i, j], est.directed[j, i]
            if e_ij and not e_ji:
                correct += t_ij
                reversed_ += t_ji
            elif e_ji and not e_ij:
                correct += t_ji
                reversed_ += t_ij
            else:
                undecided += 1
    decided = total - undecided
    return {"orient_total": total, "orient_correct": correct,
            "orient_reversed": reversed_, "orient_undecided": undecided,
            # 全体に対する正解率. 向きを出さない手法 (PAG の丸印) は低く出る
            "orient_accuracy": correct / total if total else float("nan"),
            # 向きを決めた割合と, 決めたもののなかでの正解率.
            # FCI のように丸印を残す手法を不当に低く評価しないために分ける
            "orient_decided_rate": decided / total if total else float("nan"),
            "orient_accuracy_decided": correct / decided if decided else float("nan")}


def cycle_scores(est: Estimate, truth: Estimate) -> dict:
    """巡回の再現. 真の 2-cycle を両方向の矢として出せたか, 巡回を出せたか."""
    d = len(truth.names)
    true_2c = [(i, j) for i in range(d) for j in range(i + 1, d)
               if truth.directed[i, j] and truth.directed[j, i]]
    hit = sum(1 for i, j in true_2c if est.directed[i, j] and est.directed[j, i])
    has_cycle = _has_cycle(est.directed)
    return {"true_2cycles": len(true_2c), "recovered_2cycles": hit,
            "cycle_recall": hit / len(true_2c) if true_2c else float("nan"),
            "estimate_has_cycle": has_cycle,
            "true_has_cycle": _has_cycle(truth.directed)}


def _has_cycle(A: np.ndarray) -> bool:
    """有向グラフに巡回があるか (2-cycle を含む)."""
    d = A.shape[0]
    reach = A.copy()
    for _ in range(d):
        reach = reach | (reach @ A).astype(bool)
    return bool(np.any(np.diag(reach)))


def score_all(est: Estimate, truth: Estimate) -> dict:
    return {**skeleton_scores(est, truth), **orientation_scores(est, truth),
            **cycle_scores(est, truth)}

### 2.2 生成器 (generator)

巡回構造と欠測機構を独立に制御する。rho はスペクトル半径。欠測機構は MCAR / MAR / 自己マスキング /
自己+別の変数 (mnar_mixed)。

In [ ]:
%%module generator
"""巡回構造と欠測機構を独立に制御する合成データ生成器.

卒業研究「欠測下で巡回構造を許す制約ベース因果探索アルゴリズムの提案」の検証環境の中核.
中間発表 (2026-08-06) スライド7の4つの評価軸
  (1) 巡回の複雑さ : 非巡回 / 2-cycle / 長さ3
  (2) フィードバック利得 : rho = 0.1 〜 0.9
  (3) 欠測機構 : MCAR / MAR / MNAR
  (4) 欠測率と位置 : 5〜60%, 巡回内 / 巡回外
を, 生成器のパラメータとして分離して振れるようにする.

rho の定義 (発表資料では未定義だった点):
  線形 SEM  x = B x + e  において, 長さ k の巡回を1つだけ持つ B の非零固有値は
  その巡回の辺係数の積 p (ループ利得) に対し lambda^k = p を満たす.
  よってスペクトル半径は rho(B) = |p|^(1/k) であり, 巡回の外側にある非巡回な辺を
  いくら足しても rho(B) は変わらない (それらは冪零成分にしか寄与しない).
  非巡回のとき B は冪零なので rho(B) = 0 であり, 「非巡回」は rho 軸上の内点ではない.

  本実装は rho = スペクトル半径 を第一の制御量とし,
    ループ利得 p = (符号) * rho**k,  作用素ノルム ||B||_2
  は従属量として記録する. 作用素ノルムを制御量にすると, 同じ「利得」でも巡回長で
  意味が変わり, かつ非巡回でも 0 にならないため軸の解釈が壊れる
  (NODAGS-Flow の再現実験で実際に踏んだ問題).

  rho は平衡の存在条件でもある. 反復 x <- Bx + e が収束するのは rho < 1 のときで,
  rho >= 1 では発散する. 評価計画の rho = 0.9 は安定領域の端であり, 0.95 以上を
  入れるなら「平衡が存在しない領域」を別扱いする必要がある.

変数と構造 (図1「対象データにおける因果構造」に対応):
    Z            外生の共変量. 巡回の外側. 完全観測. MAR の駆動変数に使う.
    L            未観測交絡 (出展者の認知チャネル). latent=True で観測から落とす.
    X1, X2, X3   巡回に入るノード (掲載順位, 応募数, ...). cycle により本数が変わる.
    Y            巡回の子 (開催成果). 自己マスキング型欠測の対象.
辺:
    Z  -> X1
    L  -> X1, L -> X2          (latent=True のとき L は観測から落ちる)
    巡回 : 非巡回なら X1 -> X2, 2-cycle なら X1 <-> X2, 長さ3なら X1->X2->X3->X1
    Xi -> Y                    (巡回ノードすべてから Y へ)
"""
from __future__ import annotations

from dataclasses import dataclass, field, asdict

import numpy as np

# "acyclic" は後方互換のための別名で, 中身は acyclic2 (X1 -> X2 の鎖) である.
# 巡回の有無だけを差にした対照を取るため, 長さ3の巡回には変数の本数をそろえた
# acyclic3 (X1 -> X2 -> X3 の鎖, 閉じる辺だけが無い) を用意する.
CYCLE_KINDS = ("acyclic", "acyclic2", "acyclic3", "2-cycle", "3-cycle")
ACYCLIC_KINDS = ("acyclic", "acyclic2", "acyclic3")
# mnar_mixed は, 欠測確率が当該変数自身と別の観測変数の両方で決まる型.
# MVPC の仮定4 が除外するのは自己マスキング一般だが, 同論文の付録 A.1 は
# 線形ガウスなら「欠測指標が当該変数以外の親を持つとき」に限って結果に影響すると述べている.
# その場合を作るための機構である.
MECHANISMS = ("mcar", "mar", "mnar_self", "mnar_mixed")
NOISES = ("gaussian", "uniform", "laplace", "gumbel")


# ---------------------------------------------------------------------------
# 設定
# ---------------------------------------------------------------------------
@dataclass
class MissingSpec:
    """1変数に掛ける欠測機構.

    target     : 欠測させる変数名
    mechanism  : "mcar" (完全無作為) / "mar" (他の完全観測変数に依存) / "mnar_self" (自分の値に依存)
    rate       : 目標欠測率. 切片を二分法で較正して実測率をこれに合わせる
    driver     : mar のとき欠測確率を駆動する変数名. 完全観測でなければならない
    slope      : ロジスティックの傾き. 大きいほど機構が強い (0 にすると MCAR に退化)
    direction  : +1 なら値が大きいほど欠測しやすい, -1 なら小さいほど欠測しやすい.
                 図1 の「成果が振るわないほど報告されない」は -1 に対応する
    """

    target: str
    mechanism: str = "mcar"
    rate: float = 0.3
    driver: str | None = None
    slope: float = 3.0
    direction: int = -1

    def __post_init__(self):
        if self.mechanism not in MECHANISMS:
            raise ValueError(f"未知の欠測機構: {self.mechanism}")
        if self.mechanism in ("mar", "mnar_mixed") and self.driver is None:
            raise ValueError(f"{self.mechanism} には driver が必要")
        if not 0.0 <= self.rate < 1.0:
            raise ValueError("rate は [0,1)")


@dataclass
class Scenario:
    """生成する1条件."""

    cycle: str = "2-cycle"          # 巡回の複雑さの軸
    rho: float = 0.6                # フィードバック利得の軸 (スペクトル半径)
    loop_sign: int = 1              # ループ利得の符号
    latent: bool = True             # 未観測交絡 L を隠すか
    confound: str = "both"          # L がどの巡回ノードに入るか: both / none / X1 / X2
    y_parent: bool = False          # Y に, 巡回と独立な外生の親 V を足すか
    missing: list[MissingSpec] = field(default_factory=list)
    noise: str = "gaussian"
    noise_scale: float = 1.0
    standardize: bool = False       # 生成後に各列を標準化するか (varsortability を消す)
    dag_coef: float = 0.8           # 巡回に入らない辺の係数の大きさ
    n_extra: int = 0                # 補助変数の本数. 骨格の精度に余地を作るために足す
    extra_density: float = 0.25     # 補助変数がらみの辺の生成確率
    seed: int = 0

    def __post_init__(self):
        if self.cycle not in CYCLE_KINDS:
            raise ValueError(f"未知の巡回種別: {self.cycle}")
        if self.noise not in NOISES:
            raise ValueError(f"未知のノイズ: {self.noise}")
        if self.cycle not in ACYCLIC_KINDS and not 0.0 < self.rho:
            raise ValueError("巡回ありのとき rho > 0")

    @property
    def cycle_len(self) -> int:
        return {"acyclic": 0, "acyclic2": 0, "acyclic3": 0,
                "2-cycle": 2, "3-cycle": 3}[self.cycle]

    @property
    def n_cycle_nodes(self) -> int:
        """巡回の候補になるノードの本数. 非巡回の対照は本数をそろえる."""
        return {"acyclic": 2, "acyclic2": 2, "acyclic3": 3,
                "2-cycle": 2, "3-cycle": 3}[self.cycle]

    @property
    def control(self) -> str:
        """この条件と変数の本数をそろえた非巡回の対照の名前."""
        return "acyclic2" if self.n_cycle_nodes == 2 else "acyclic3"


# ---------------------------------------------------------------------------
# 構造行列
# ---------------------------------------------------------------------------
def variables(cycle: str, n_extra: int = 0, y_parent: bool = False
              ) -> tuple[list[str], list[str]]:
    """(全変数, 巡回の候補になる変数) を返す. 順序は位相順序になっている.

    補助変数 W は巡回ブロックの前半 (上流) と後半 (下流) に分けて置く.
    位相順序を守って辺を張るので, 生成されるグラフの巡回は planted な1本だけになり,
    スペクトル半径 = 目標 rho が保たれる.
    """
    three = cycle in ("3-cycle", "acyclic3")
    xs = ["X1", "X2", "X3"] if three else ["X1", "X2"]
    n_pre = n_extra // 2
    pre = [f"Wu{i+1}" for i in range(n_pre)]
    post = [f"Wd{i+1}" for i in range(n_extra - n_pre)]
    head = ["Z", "L", "V"] if y_parent else ["Z", "L"]
    return [*head, *pre, *xs, "Y", *post], xs


def build_B(sc: Scenario) -> tuple[np.ndarray, list[str], list[str]]:
    """係数行列 B を作る. B[i, j] は x_j -> x_i の係数 (x = Bx + e).

    巡回の辺はすべて大きさ rho にする. 長さ k の巡回で各辺が rho なら
    ループ利得は ±rho**k, 非零固有値は |lambda| = rho となり, スペクトル半径が
    ちょうど rho になる. 巡回外の辺は dag_coef で固定し, 符号だけ交互にする.
    """
    names, cyc = variables(sc.cycle, sc.n_extra, sc.y_parent)
    idx = {v: i for i, v in enumerate(names)}
    d = len(names)
    B = np.zeros((d, d))

    c = sc.dag_coef
    B[idx["X1"], idx["Z"]] = c            # Z -> X1
    if sc.confound in ("both", "X1"):
        B[idx["X1"], idx["L"]] = c        # L -> X1
    if sc.confound in ("both", "X2"):
        B[idx["X2"], idx["L"]] = -c       # L -> X2
    for v in cyc:
        B[idx["Y"], idx[v]] = c           # Xi -> Y
    if sc.y_parent:
        B[idx["Y"], idx["V"]] = c         # V -> Y. V は巡回とは独立な外生変数

    k = sc.cycle_len
    if k == 0:
        # 非巡回の対照: 巡回ノードを鎖でつなぎ, 閉じる辺だけを持たない
        for a in range(len(cyc) - 1):
            B[idx[cyc[a + 1]], idx[cyc[a]]] = c
    else:
        # 巡回 X1 -> X2 -> ... -> Xk -> X1 を大きさ rho の辺で作る.
        # 符号は最後の辺だけで loop_sign を作る.
        for a in range(k):
            src, dst = cyc[a], cyc[(a + 1) % k]
            w = sc.rho
            if a == k - 1 and sc.loop_sign < 0:
                w = -w
            B[idx[dst], idx[src]] = w

    if sc.n_extra > 0:
        rng = np.random.default_rng(10_000 + sc.seed)
        block = {idx[v] for v in cyc}
        # 位相順序 (names の並び) の前から後ろへだけ辺を張る.
        # 巡回ブロック内部は既に決まっているので触らない.
        for a in range(d):
            for b in range(a + 1, d):
                if a in block and b in block:
                    continue
                if names[a] == "L" and names[b] not in cyc:
                    continue          # 未観測交絡は巡回ノードにだけ効かせる
                if names[a] == "V" or names[b] == "V":
                    continue          # V は Y の親としてだけ使う
                if B[b, a] != 0.0:
                    continue
                if rng.random() < sc.extra_density:
                    B[b, a] = c * rng.choice([-1.0, 1.0])
    return B, names, cyc


def spectral_radius(B: np.ndarray) -> float:
    return float(np.max(np.abs(np.linalg.eigvals(B))))


def loop_gain(B: np.ndarray, cyc_idx: list[int]) -> float:
    """巡回に沿った係数の積. cyc_idx は巡回の順序で並んだ添字."""
    k = len(cyc_idx)
    if k == 0:
        return 0.0
    p = 1.0
    for a in range(k):
        src, dst = cyc_idx[a], cyc_idx[(a + 1) % k]
        p *= B[dst, src]
    return float(p)


# ---------------------------------------------------------------------------
# サンプリング
# ---------------------------------------------------------------------------
def draw_noise(kind: str, scale: float, size, rng: np.random.Generator) -> np.ndarray:
    """分散が scale**2 になるよう規格化した誤差を引く."""
    if kind == "gaussian":
        return rng.normal(0.0, scale, size=size)
    if kind == "uniform":
        half = np.sqrt(3.0) * scale
        return rng.uniform(-half, half, size=size)
    if kind == "laplace":
        return rng.laplace(0.0, scale / np.sqrt(2.0), size=size)
    if kind == "gumbel":
        beta = scale * np.sqrt(6.0) / np.pi
        g = rng.gumbel(0.0, beta, size=size)
        return g - beta * np.euler_gamma
    raise ValueError(kind)


def standardize_columns(X: np.ndarray) -> np.ndarray:
    """各列を平均0・分散1にそろえる.

    誤差分散をそろえたまま線形 SEM を作ると, 下流の変数ほど周辺分散が大きくなる.
    すると「分散の小さい順に並べる」だけで因果順序が当たってしまい,
    手法の性能を過大評価する (varsortability, Reisach ら 2021 の指摘).
    実測でも, 標準化しない既定の生成器は varsortability が 1.000 になる.

    誤差分散の側で周辺分散を 1 にそろえる手もあるが, 本生成器では辺の係数が密なため
    正の解を持たない (親からの寄与だけで分散が 1 を超える). 係数を縮めれば解けるが,
    巡回の辺の大きさは rho そのものなので, 縮めると rho の制御が壊れる.
    したがってデータ側を標準化する. 偏相関は尺度不変なので制約ベースの手法の
    判定は変わらず, 分散の並びを手がかりにできる手法からその手がかりだけを取り除ける.
    """
    mu = np.mean(X, axis=0)
    sd = np.std(X, axis=0)
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mu) / sd


def varsortability(X: np.ndarray, B: np.ndarray) -> float:
    """分散の並びだけで因果の向きがどれだけ当たるか. 1 に近いほど人工物が強い.

    有向辺 j -> i について Var(x_j) < Var(x_i) なら 1, 等しければ 0.5 と数える.
    """
    v = np.var(X, axis=0)
    hit = tie = tot = 0
    d = B.shape[0]
    for i in range(d):
        for j in range(d):
            if i == j or B[i, j] == 0:
                continue
            tot += 1
            if v[j] < v[i]:
                hit += 1
            elif np.isclose(v[j], v[i]):
                tie += 1
    return float("nan") if tot == 0 else (hit + 0.5 * tie) / tot


def sample_equilibrium(B: np.ndarray, E: np.ndarray) -> np.ndarray:
    """平衡 x = (I - B)^{-1} e を閉形式で解く."""
    d = B.shape[0]
    return E @ np.linalg.inv(np.eye(d) - B).T


def sample_iterate(B: np.ndarray, E: np.ndarray, n_iter: int = 500,
                   tol: float = 1e-10) -> tuple[np.ndarray, int, bool]:
    """反復 x <- Bx + e で平衡に寄せる. (x, 反復回数, 収束したか) を返す.

    rho(B) < 1 なら収束する. 平衡の存在条件を数値で見せるために閉形式とは別に持つ.
    """
    x = np.zeros_like(E)
    for t in range(1, n_iter + 1):
        x_new = x @ B.T + E
        delta = float(np.max(np.abs(x_new - x)))
        x = x_new
        if not np.isfinite(delta):
            return x, t, False
        if delta < tol:
            return x, t, True
    return x, n_iter, False


# ---------------------------------------------------------------------------
# 欠測機構
# ---------------------------------------------------------------------------
def _sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


def _calibrate_intercept(score: np.ndarray, rate: float) -> float:
    """平均欠測確率が rate になる切片を二分法で求める."""
    lo, hi = -80.0, 80.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        p = float(np.mean(_sigmoid(score + mid)))
        if p < rate:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def missing_probability(X: np.ndarray, names: list[str], spec: MissingSpec
                        ) -> tuple[np.ndarray, dict]:
    """欠測確率ベクトル p (n,) と較正情報を返す. 1 が欠測."""
    idx = {v: i for i, v in enumerate(names)}
    n = X.shape[0]
    if spec.mechanism == "mcar":
        return np.full(n, spec.rate), {"intercept": None, "driver": None}

    def std(name):
        v = X[:, idx[name]]
        sd = float(np.std(v))
        return (v - float(np.mean(v))) / (sd if sd > 0 else 1.0)

    if spec.mechanism == "mnar_mixed":
        # 自分の値と駆動変数の両方に依存させる. 係数は半分ずつにして強さをそろえる
        z = spec.slope * spec.direction * 0.5 * (std(spec.target) + std(spec.driver))
        src = f"{spec.target}+{spec.driver}"
    else:
        src = spec.target if spec.mechanism == "mnar_self" else spec.driver
        z = spec.slope * spec.direction * std(src)
    b = _calibrate_intercept(z, spec.rate)
    return _sigmoid(z + b), {"intercept": b, "driver": src}


def apply_missingness(X: np.ndarray, names: list[str], specs: list[MissingSpec],
                      rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray, dict]:
    """X に欠測を入れる. (欠測入り X, 欠測指標 M, 較正情報) を返す. M=1 が欠測."""
    idx = {v: i for i, v in enumerate(names)}
    Xm = X.copy()
    M = np.zeros_like(X, dtype=bool)
    info = {}
    for spec in specs:
        if spec.target not in idx:
            raise ValueError(f"{spec.target} は変数に無い")
        p, meta = missing_probability(X, names, spec)
        m = rng.random(X.shape[0]) < p
        M[:, idx[spec.target]] = m
        Xm[m, idx[spec.target]] = np.nan
        info[spec.target] = {**meta, "mechanism": spec.mechanism,
                             "target_rate": spec.rate, "realized_rate": float(m.mean())}
    return Xm, M, info


# ---------------------------------------------------------------------------
# 入口
# ---------------------------------------------------------------------------
def generate(sc: Scenario, n: int, seed: int | None = None) -> dict:
    """1条件ぶんのデータを作る.

    返り値の主なキー
      X_full   : 欠測前・全変数 (未観測交絡 L を含む)
      X_obs    : 観測される行列. latent=True なら L の列を落とし, 欠測は NaN
      obs_names: X_obs の列名
      M        : 欠測指標 (X_full と同じ形, True が欠測)
      B        : 係数行列 (全変数)
      B_obs    : 観測変数だけに落とした真の隣接 (辺の有無の真値として使う)
      meta     : rho の実測値・ループ利得・作用素ノルム・欠測率など
    """
    rng = np.random.default_rng(sc.seed if seed is None else seed)
    B, names, cyc = build_B(sc)
    idx = {v: i for i, v in enumerate(names)}
    E = draw_noise(sc.noise, sc.noise_scale, (n, len(names)), rng)
    X = sample_equilibrium(B, E)
    if sc.standardize:
        X = standardize_columns(X)
    Xm, M, minfo = apply_missingness(X, names, sc.missing, rng)

    drop = [idx["L"]] if sc.latent else []
    keep = [i for i in range(len(names)) if i not in drop]
    obs_names = [names[i] for i in keep]

    cyc_idx = [idx[v] for v in cyc]
    meta = {
        "scenario": asdict(sc),
        "names": names,
        "cycle_nodes": cyc,
        "n": n,
        "rho_target": 0.0 if sc.cycle in ACYCLIC_KINDS else sc.rho,
        "rho_actual": spectral_radius(B),
        "loop_gain": loop_gain(B, cyc_idx),
        "op_norm": float(np.linalg.norm(B, 2)),
        "missing": minfo,
        # rho を上げると平衡の分散が発散する. 手法の劣化が「巡回の強さ」由来なのか
        # 「分散の増大」由来なのかを後で分離できるよう, 周辺分散を必ず記録する.
        "marginal_var": {v: float(np.var(X[:, idx[v]])) for v in names},
        "varsortability": varsortability(X, B),
    }
    return {
        "X_full": X,
        "X_obs": Xm[:, keep],
        "obs_names": obs_names,
        "M": M,
        "B": B,
        "B_obs": B[np.ix_(keep, keep)],
        "E": E,
        "meta": meta,
    }

### 2.3 巡回グラフの d分離 (dsep)

In [ ]:
%%module dsep
"""巡回を含む有向グラフの d分離を計算し, 巡回と非巡回が区別できるかをグラフの水準で判定する.

diagnostics.py の D6 は母共分散の偏相関で条件付き独立を判定していた。
これは線形ガウスに限った話に見えるが, 線形の巡回 SCM では d分離に関する
大域的有向マルコフ性が成り立つ (Spirtes 1995) ので, 独立の有無はグラフだけで決まる。
したがって, 2つのグラフが観測変数上で同じ d分離を持つなら,
どんな独立性検定を使っても (相関でも HSIC でも KCI でも) 区別できない。

ここでは d分離を直接計算して, 偏相関で見た一致がグラフの性質であることを確かめる。
非線形の場合は d分離ではなく sigma分離が要るので, この結論は線形に限る。

実行:  uv run python src/dsep.py
"""
from __future__ import annotations

import itertools
import os
import sys



def descendants(edges: set[tuple[str, str]], v: str) -> set[str]:
    """v から有向路で到達できる点 (v 自身を含む). 巡回があっても止まる."""
    seen = {v}
    stack = [v]
    while stack:
        u = stack.pop()
        for a, b in edges:
            if a == u and b not in seen:
                seen.add(b)
                stack.append(b)
    return seen


def _edge_paths(adj: dict[str, set[str]], edges: set[tuple[str, str]],
                src: str, dst: str):
    """向きを無視した単純路を, 使う辺の向きまで区別して列挙する.

    2-cycle があると X1 と X2 の間に両方の辺があるので, 同じ頂点の並びでも
    「X1 -> X2 を使う路」と「X2 -> X1 を使う路」は別の路になる.
    頂点の並びだけで数えると, 合流点の判定を取り違える.
    返すのは (頂点列, 向き列) で, 向き列の k 番目は
    True なら path[k] -> path[k+1], False なら path[k+1] -> path[k] を表す.
    """
    path = [src]
    dirs: list[bool] = []
    seen = {src}

    def walk(u):
        if u == dst:
            yield list(path), list(dirs)
            return
        for w in sorted(adj[u]):
            if w in seen:
                continue
            for forward in (True, False):
                if forward and (u, w) not in edges:
                    continue
                if not forward and (w, u) not in edges:
                    continue
                seen.add(w)
                path.append(w)
                dirs.append(forward)
                yield from walk(w)
                dirs.pop()
                path.pop()
                seen.remove(w)

    yield from walk(src)


def d_separated(nodes: list[str], edges: set[tuple[str, str]], i: str, j: str,
                cond: tuple[str, ...]) -> bool:
    """巡回を許す有向グラフでの d分離.

    線形の巡回 SCM では d分離に関する大域的有向マルコフ性が成り立つ (Spirtes 1995)。
    非線形では sigma分離が要るので, この判定は線形に限る。
    """
    adj: dict[str, set[str]] = {v: set() for v in nodes}
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)
    S = set(cond)
    desc = {v: descendants(edges, v) for v in nodes}

    for path, dirs in _edge_paths(adj, edges, i, j):
        blocked = False
        for k in range(1, len(path) - 1):
            mid = path[k]
            # dirs[k-1] が True なら path[k-1] -> mid, False なら mid -> path[k-1]
            into_mid_from_prev = dirs[k - 1]
            # dirs[k] が True なら mid -> path[k+1], False なら path[k+1] -> mid
            into_mid_from_next = not dirs[k]
            if into_mid_from_prev and into_mid_from_next:      # 合流点
                if not (desc[mid] & S):
                    blocked = True
                    break
            else:                                              # 非合流点
                if mid in S:
                    blocked = True
                    break
        if not blocked:
            return False
    return True


def dsep_signature(nodes: list[str], edges: set[tuple[str, str]],
                   observed: list[str]) -> set[str]:
    sig = set()
    for i, j in itertools.combinations(observed, 2):
        rest = [v for v in observed if v not in (i, j)]
        for r in range(len(rest) + 1):
            for cond in itertools.combinations(rest, r):
                if d_separated(nodes, edges, i, j, cond):
                    sig.add(f"{i} _||_ {j} | {','.join(cond) or '{}'}")
    return sig


def main() -> None:
    nodes = ["Z", "L", "X1", "X2", "Y"]
    observed = ["Z", "X1", "X2", "Y"]
    base = {("Z", "X1"), ("X1", "Y"), ("X2", "Y")}
    confounds = {"あり (両方)": {("L", "X1"), ("L", "X2")},
                 "片方 (X1のみ)": {("L", "X1")},
                 "なし": set()}

    print("d分離の集合が一致するか (観測変数 Z, X1, X2, Y)")
    print("一致するなら, どんな独立性検定を使っても区別できない\n")
    print(f"{'未観測交絡':<14}{'一致':<7}{'非巡回のd分離数':>16}{'2-cycleのd分離数':>17}")
    for label, conf in confounds.items():
        acy = base | conf | {("X1", "X2")}
        cyc = base | conf | {("X1", "X2"), ("X2", "X1")}
        sa = dsep_signature(nodes, acy, observed)
        sc = dsep_signature(nodes, cyc, observed)
        print(f"{label:<14}{str(sa == sc):<7}{len(sa):>16}{len(sc):>17}")
        if sa != sc:
            only_a = sorted(sa - sc)
            only_c = sorted(sc - sa)
            if only_a:
                print(f"    非巡回のみ: {only_a}")
            if only_c:
                print(f"    巡回のみ  : {only_c}")

    # 逆向きの非巡回とも比べる
    print("\n両方の向きの非巡回と比べたとき (2-cycle が本当に区別できるか)")
    for label, conf in confounds.items():
        cyc = base | conf | {("X1", "X2"), ("X2", "X1")}
        sc = dsep_signature(nodes, cyc, observed)
        same = []
        for name, e in (("X1->X2", {("X1", "X2")}), ("X2->X1", {("X2", "X1")})):
            sa = dsep_signature(nodes, base | conf | e, observed)
            if sa == sc:
                same.append(name)
        verdict = "区別できない (" + ",".join(same) + " と同一)" if same else "区別できる"
        print(f"  未観測交絡 {label:<14} {verdict}")


if __name__ == "__main__":
    main()

### 2.4 既存手法のラッパ (methods)

9月の環境で使った実装 (causal-learn, lingam, 自作の LiNG-D, 自明なベースライン)。
run_mvpc_fixed は causal-learn の MVPC から、欠測指標の親の検出の誤りだけを直したもの。

In [ ]:
%%module methods
"""既存手法のラッパ. 欠測入りの行列を渡すと共通表現の推定結果を返す.

中間発表スライド5・7で比較対象に挙げた手法のうち, 公開実装で動かせるものを揃える.
どの手法が何を仮定しているかを, 実装の側でも明示しておく.

  pc_cc        PC (行ごと削除)         非巡回・因果十分性・欠測は MCAR でないと歪む
  pc_impute    PC (平均代入)           代入は MCAR 以外で歪む. 素朴な対処の対照
  mvpc         MVPC (補正付き検定)     非巡回・自己マスキングなし (Tu et al. 2019)
  fci_cc       FCI (行ごと削除)        非巡回だが潜在交絡は扱える
  direct_lingam DirectLiNGAM           非巡回・線形非ガウス・因果十分性
  ica_lingam    ICA-LiNGAM             非巡回・線形非ガウス・因果十分性
  rcd           RCD                    非巡回・線形非ガウス・潜在交絡あり
  mlingam       m-LiNGAM               欠測を扱う LiNGAM (lingam 1.12 に同梱)
  corr_thresh   相関の閾値              パラメータを持たない自明なベースライン

自明なベースラインを必ず併記するのは, 以前に行った NN による因果探索の検証で,
手法の見かけの優劣が実装や指標の人工物に由来する場合があると確認したため.
「破綻」はこのベースラインに負けることとして操作的に定義する.

CCD と LiNG-D と MissDAG と MissNODAG は Python の公開実装が手元に無いので,
ここには入っていない. 公式実装のラッパは official_tetrad と official_missing にある.
"""
from __future__ import annotations

import os
import sys
import warnings

import numpy as np

from metrics import Estimate, from_causal_learn, from_weight_matrix

WEIGHT_THRESH = 0.1
MAX_PERM_DIM = 7        # これを超えたら並べ替えの全列挙をやめて候補を絞る


def _complete_case(X: np.ndarray) -> np.ndarray:
    return X[~np.isnan(X).any(axis=1)]


def _mean_impute(X: np.ndarray) -> np.ndarray:
    Xi = X.copy()
    for j in range(X.shape[1]):
        col = Xi[:, j]
        m = np.isnan(col)
        if m.any():
            col[m] = np.nanmean(col)
    return Xi


# ---------------------------------------------------------------------------
def run_pc_cc(X, names, alpha=0.05, **kw) -> Estimate:
    from causallearn.search.ConstraintBased.PC import pc
    cg = pc(_complete_case(X), alpha=alpha, indep_test="fisherz",
            show_progress=False, node_names=names)
    return from_causal_learn(cg.G.graph, names, "cpdag")


def run_pc_impute(X, names, alpha=0.05, **kw) -> Estimate:
    from causallearn.search.ConstraintBased.PC import pc
    cg = pc(_mean_impute(X), alpha=alpha, indep_test="fisherz",
            show_progress=False, node_names=names)
    return from_causal_learn(cg.G.graph, names, "cpdag")


def run_mvpc(X, names, alpha=0.05, **kw) -> Estimate:
    from causallearn.search.ConstraintBased.PC import pc
    cg = pc(X, alpha=alpha, mvpc=True, correction_name="MV_Crtn_Fisher_Z",
            show_progress=False, node_names=names)
    return from_causal_learn(cg.G.graph, names, "cpdag")


def _fixed_parent_missingness_pairs(data, alpha, indep_test, stable=True, max_k=None):
    """causal-learn 0.1.4.8 の get_parent_missingness_pairs の修正版.

    元の detect_parent は, 列 r を欠測指標に置き換えたデータの複製を作るが,
    検定には置き換える前のデータで作った CIT をそのまま使う. そのため
    「R_r と y が独立か」ではなく「X_r の値と y が独立か」を検定してしまい,
    欠測指標の親ではなく X_r の隣接点を親として返す.
    公式 R 実装 (detection_prt_m) は欠測指標そのものを検定している.
    ここでは指標に置き換えたデータで CIT を作り直す. それ以外は元の関数と同じ.
    """
    from causallearn.search.ConstraintBased import PC as clpc
    from causallearn.utils.cit import CIT
    out = {"prt": [], "m": []}
    for r in clpc.get_missingness_index(data):
        Xr = data.copy()
        Xr[:, r] = np.isnan(data[:, r]).astype(float)
        test = CIT(Xr, indep_test.method)
        prt = clpc.detect_parent(r, data, alpha, test, stable, max_k=max_k)
        if len(prt):
            out["prt"].append(prt)
            out["m"].append(r)
    return out


def run_mvpc_fixed(X, names, alpha=0.05, **kw) -> Estimate:
    """causal-learn の MVPC から, 欠測指標の親の検出の誤りだけを直したもの."""
    from causallearn.search.ConstraintBased import PC as clpc
    orig = clpc.get_parent_missingness_pairs
    clpc.get_parent_missingness_pairs = _fixed_parent_missingness_pairs
    try:
        cg = clpc.pc(X, alpha=alpha, mvpc=True, correction_name="MV_Crtn_Fisher_Z",
                     show_progress=False, node_names=names)
    finally:
        clpc.get_parent_missingness_pairs = orig
    return from_causal_learn(cg.G.graph, names, "cpdag")


def run_fci_cc(X, names, alpha=0.05, **kw) -> Estimate:
    from causallearn.search.ConstraintBased.FCI import fci
    G, _ = fci(_complete_case(X), independence_test_method="fisherz", alpha=alpha,
               show_progress=False, verbose=False, node_names=names)
    return from_causal_learn(G.graph, names, "pag")


def run_direct_lingam(X, names, **kw) -> Estimate:
    import lingam
    m = lingam.DirectLiNGAM()
    m.fit(_complete_case(X))
    return from_weight_matrix(m.adjacency_matrix_, names, WEIGHT_THRESH, "dag")


def run_ica_lingam(X, names, **kw) -> Estimate:
    import lingam
    m = lingam.ICALiNGAM()
    m.fit(_complete_case(X))
    return from_weight_matrix(m.adjacency_matrix_, names, WEIGHT_THRESH, "dag")


def run_rcd(X, names, **kw) -> Estimate:
    import lingam
    m = lingam.RCD()
    m.fit(_complete_case(X))
    # RCD は潜在交絡が疑われる対を NaN で返す. 辺としては採らないが双方向として記録する
    W = m.adjacency_matrix_
    conf = np.isnan(W)
    est = from_weight_matrix(np.nan_to_num(W), names, WEIGHT_THRESH, "dag+latent")
    bi = (conf | conf.T)
    np.fill_diagonal(bi, False)
    est.bidirected = bi
    est.adjacent = est.adjacent | bi
    return est


_LINGAM_PATCHED = False


def _patch_lingam_logistic() -> None:
    """lingam 1.12.2 の m-LiNGAM を scikit-learn 1.8 で動かすための互換処理.

    sklearn 1.8 で私的関数 _logistic_regression_path に classes がキーワード必須で
    追加されたが, lingam 側は渡していないので TypeError で落ちる.
    呼び出し規約を変えずに classes を補う. 中身の計算には手を入れていない.
    """
    global _LINGAM_PATCHED
    if _LINGAM_PATCHED:
        return
    import inspect

    import lingam.utils as lu
    from sklearn.linear_model._logistic import _logistic_regression_path as orig

    if "classes" in inspect.signature(orig).parameters:
        def patched(X, y, **kwargs):
            kwargs.setdefault("classes", np.unique(y))
            return orig(X, y, **kwargs)
        lu._logistic_regression_path = patched
    _LINGAM_PATCHED = True


def run_mlingam(X, names, **kw) -> Estimate:
    import lingam
    _patch_lingam_logistic()
    m = lingam.mLiNGAM()
    m.fit(X)
    return from_weight_matrix(m.adjacency_matrix_, names, WEIGHT_THRESH, "dag")


def run_corr_thresh(X, names, thresh=0.2, **kw) -> Estimate:
    """パラメータを1つしか持たない自明なベースライン. 対ごとに使える行だけで相関を取る."""
    d = X.shape[1]
    A = np.zeros((d, d), bool)
    for i in range(d):
        for j in range(i + 1, d):
            ok = ~np.isnan(X[:, i]) & ~np.isnan(X[:, j])
            if ok.sum() < 10:
                continue
            r = np.corrcoef(X[ok, i], X[ok, j])[0, 1]
            if abs(r) > thresh:
                A[i, j] = A[j, i] = True
    return Estimate(names, np.zeros((d, d), bool), A, kind="skeleton")


def run_complete_graph(X, names, **kw) -> Estimate:
    """全対をつなぐ自明な骨格. 真のグラフが密なとき, これが強くなりうる."""
    d = X.shape[1]
    A = np.ones((d, d), bool)
    np.fill_diagonal(A, False)
    return Estimate(names, np.zeros((d, d), bool), A, kind="skeleton")


METHODS = {
    "pc_cc": run_pc_cc,
    "pc_impute": run_pc_impute,
    "mvpc": run_mvpc,
    "fci_cc": run_fci_cc,
    "direct_lingam": run_direct_lingam,
    "ica_lingam": run_ica_lingam,
    "rcd": run_rcd,
    "mlingam": run_mlingam,
    "corr_thresh": run_corr_thresh,
    "complete_graph": run_complete_graph,
}

# 各手法が仮定しているもの. 表の作成と考察に使う
ASSUMPTIONS = {
    "pc_cc": "非巡回, 因果十分性, 欠測は MCAR",
    "pc_impute": "非巡回, 因果十分性, 欠測は MCAR",
    "mvpc": "非巡回, 自己マスキングなし",
    "fci_cc": "非巡回, 潜在交絡可, 欠測は MCAR",
    "direct_lingam": "非巡回, 線形非ガウス, 因果十分性",
    "ica_lingam": "非巡回, 線形非ガウス, 因果十分性",
    "rcd": "非巡回, 線形非ガウス, 潜在交絡可",
    "mlingam": "非巡回, 線形非ガウス, 欠測機構をモデル化",
    "corr_thresh": "仮定なし (骨格のみ)",
    "complete_graph": "仮定なし (全対をつなぐ)",
}


def run_method(name: str, X: np.ndarray, names: list[str], **kw):
    """例外を握りつぶさずに (推定, エラー) の組で返す.

    一部の実装が推定した辺を標準出力に書くので, ログが混ざらないよう捨てる.
    """
    import contextlib
    import io

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf):
                return METHODS[name](X, names, **kw), None
        except Exception as exc:  # noqa: BLE001
            return None, f"{type(exc).__name__}: {exc}"


# ---------------------------------------------------------------------------
# LiNG-D (Lacerda, Spirtes, Ramsey, Hoyer, UAI 2008) の最小実装
# ---------------------------------------------------------------------------
def ling_d(X: np.ndarray, n_restart: int = 3, tol: float = 1e-8,
           seed: int = 0) -> dict:
    """線形非ガウスの巡回モデルを ICA から推定する.

    x = Bx + e から x = (I-B)^{-1} e なので, 混合行列 A = (I-B)^{-1} を ICA で求め,
    W = A^{-1} の行を並べ替えて対角を 1 に正規化すると B = I - W~ が得られる.
    非巡回を仮定する LiNGAM は「W を下三角に並べ替えられる」ことを要求するが,
    LiNG-D はその制約を外し, 対角が非零になる並べ替えすべてを候補として残す.
    候補のうち安定なもの (スペクトル半径 < 1) が解の集合であり, 一意とは限らない.

    戻り値: {"solutions": [B, ...], "stable": [B, ...], "n_perm_valid": int}
    """
    from itertools import permutations

    from sklearn.decomposition import FastICA

    d = X.shape[1]
    best = None
    for r in range(n_restart):
        ica = FastICA(n_components=d, whiten="unit-variance", max_iter=2000,
                      tol=1e-5, random_state=seed + r)
        try:
            ica.fit(X)
        except Exception:  # noqa: BLE001
            continue
        W = ica.components_
        if best is None:
            best = W
    if best is None:
        raise RuntimeError("ICA が収束しなかった")

    # 全並べ替えの列挙は d! なので, 変数が増えたら候補を絞る.
    # 絞る場合は「解の集合」を全部見ていないので, その旨を戻り値に残す.
    if d <= MAX_PERM_DIM:
        cand_perms = list(permutations(range(d)))
        exhaustive = True
    else:
        from scipy.optimize import linear_sum_assignment
        cost = -np.log(np.abs(best) + 1e-12)
        row, col = linear_sum_assignment(cost)
        base_perm = [int(r) for _, r in sorted(zip(col, row))]
        cand = {tuple(base_perm)}
        for a in range(d):
            for b in range(a + 1, d):
                q = list(base_perm)
                q[a], q[b] = q[b], q[a]
                cand.add(tuple(q))
        cand_perms = sorted(cand)
        exhaustive = False

    sols, stable = [], []
    n_valid = 0
    for perm in cand_perms:
        Wp = best[list(perm), :]
        diag = np.diag(Wp)
        if np.any(np.abs(diag) < tol):
            continue
        n_valid += 1
        Wn = Wp / diag[:, None]
        B = np.eye(d) - Wn
        sols.append(B)
        if np.max(np.abs(np.linalg.eigvals(B))) < 1.0:
            stable.append(B)
    return {"solutions": sols, "stable": stable, "n_perm_valid": n_valid,
            "exhaustive": exhaustive}


def run_lingd(X, names, thresh=WEIGHT_THRESH, **kw) -> Estimate:
    """LiNG-D. 安定解が複数あるときは辺が最も少ないものを採る.

    「安定解が一意でない」こと自体が結果なので, 何個あったかを raw に残す.
    """
    res = ling_d(_complete_case(X))
    cands = res["stable"] or res["solutions"]
    if not cands:
        raise RuntimeError("解が得られなかった")
    B = min(cands, key=lambda M: int((np.abs(M) > thresh).sum()))
    est = from_weight_matrix(B, names, thresh, "cyclic")
    est.raw = {"n_stable": len(res["stable"]), "n_perm_valid": res["n_perm_valid"],
               "exhaustive": res["exhaustive"]}
    return est


METHODS["lingd"] = run_lingd
ASSUMPTIONS["lingd"] = "巡回可, 線形非ガウス, 因果十分性, 完全観測"

### 2.5 公式実装のラッパ: Tetrad (official_tetrad)

CCD・FCI・LiNG-D を Tetrad の jar から JPype で呼ぶ。run_oracle は d分離の神託で CCD / FCI を回す。

In [ ]:
%%module official_tetrad
"""Tetrad (CMU 公式実装) の CCD・FCI・LiNG-D を呼ぶラッパ.

中間発表スライド5の比較表のうち, CCD (Richardson 1996)・FCI (Spirtes ら 1999)・
LiNG-D (Lacerda ら 2008) は, いずれも著者グループ (CMU 哲学科) の Tetrad に実装がある.
LiNG-D の論文は再現用のコードの置き場として www.phil.cmu.edu/~tetrad/ を挙げている.
CCD と FCI の論文にはコードの所在が書かれていないが, 両手法とも著者が Tetrad の開発者であり,
Tetrad の実装が参照実装にあたる.

取得元: official/py-tetrad (https://github.com/cmu-phil/py-tetrad) に同梱の
tetrad-current.jar. 版は env_report が記録する. この jar は Java 21 以上を要するので,
official/jdk/ に置いた Temurin 21 の JVM を JPype で起動する.

欠測の扱い: CCD・FCI・LiNG-D はいずれも完全観測を前提にした手法なので,
論文どおりの使い方は行ごと削除 (listwise) である. Tetrad の Fisher Z 検定は
検定ごとの削除 (testwise) も持つので, 比較用に両方を出せるようにする.
"""
from __future__ import annotations

import glob
import os
import sys

import numpy as np

from metrics import Estimate

ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
JAR = os.path.join(ROOT, "official", "py-tetrad", "pytetrad", "resources",
                   "tetrad-current.jar")


def _libjvm() -> str:
    cands = glob.glob(os.path.join(ROOT, "official", "jdk", "jdk-21*", "Contents",
                                   "Home", "lib", "server", "libjvm.dylib"))
    cands += glob.glob(os.path.join(ROOT, "official", "jdk", "jdk-21*", "lib",
                                    "server", "libjvm.so"))
    if not cands:
        raise RuntimeError("official/jdk に Java 21 が無い. README の手順 (scripts/setup_official.sh) で入れる")
    return cands[0]


def start_jvm() -> None:
    import jpype
    if not jpype.isJVMStarted():
        jpype.startJVM(_libjvm(), "-Xmx4g", "--enable-native-access=ALL-UNNAMED",
                       classpath=[JAR], convertStrings=True)
        # Tetrad は探索中の辺を標準出力に書くので捨てる (結果は戻り値で受け取る)
        jl = jpype.JPackage("java")
        jl.lang.System.setOut(jl.io.PrintStream(jl.io.OutputStream.nullOutputStream()))


def tetrad_version() -> str:
    start_jvm()
    import jpype
    try:
        return str(jpype.JClass("edu.cmu.tetrad.util.Version").currentViewableVersion())
    except Exception:  # noqa: BLE001
        return "unknown"


def _dataset(X: np.ndarray, names: list[str]):
    start_jvm()
    import jpype
    ArrayList = jpype.JClass("java.util.ArrayList")
    td = jpype.JPackage("edu").cmu.tetrad.data
    vars_ = ArrayList()
    for nm in names:
        vars_.add(td.ContinuousVariable(nm))
    box = td.DoubleDataBox(jpype.JArray(jpype.JDouble, 2)(np.ascontiguousarray(X, float).tolist()))
    return td.BoxDataSet(box, vars_)


def _fisher_z(X, names, alpha, missing):
    import jpype
    tm = jpype.JPackage("edu").cmu.tetrad.data.missing
    tt = jpype.JPackage("edu").cmu.tetrad.search.test
    spec = {"listwise": tm.MissingDataSpec.listwise,
            "testwise": tm.MissingDataSpec.testwise}[missing]()
    return tt.IndTestFisherZ(_dataset(X, names), float(alpha), spec)


def _from_tetrad_graph(g, names: list[str], kind: str) -> Estimate:
    """Tetrad の Graph を共通表現にする.

    端点は TAIL / ARROW / CIRCLE. 矢を受ける側を見て向きを決める.
      a --> b           : 尾と矢
      a <-> b           : 矢と矢 (潜在交絡)
      a o-> b, a o-o b  : 丸印を含むので向きは未確定として扱う
    CCD の出力 (PAG) は 2-cycle を直接には表さない. 巡回は
    「X1 と X2 が隣接し, どちらの端点も矢でない」などの形で現れうるので,
    rawに辺の文字列を残して後から読めるようにする.
    """
    d = len(names)
    idx = {nm: i for i, nm in enumerate(names)}
    directed = np.zeros((d, d), bool)
    adjacent = np.zeros((d, d), bool)
    bidirected = np.zeros((d, d), bool)
    edges = []
    for e in g.getEdges():
        a, b = idx[str(e.getNode1().getName())], idx[str(e.getNode2().getName())]
        ea, eb = str(e.getEndpoint1().name()), str(e.getEndpoint2().name())
        edges.append(str(e))
        adjacent[a, b] = adjacent[b, a] = True
        if ea == "TAIL" and eb == "ARROW":
            directed[a, b] = True
        elif ea == "ARROW" and eb == "TAIL":
            directed[b, a] = True
        elif ea == "ARROW" and eb == "ARROW":
            bidirected[a, b] = bidirected[b, a] = True
    return Estimate(names, directed, adjacent, bidirected, kind=kind,
                    raw={"edges": sorted(edges)})


def _complete_case(X):
    return X[~np.isnan(X).any(axis=1)]


# ---------------------------------------------------------------------------
def run_ccd(X, names, alpha=0.05, missing="listwise", **kw) -> Estimate:
    start_jvm()
    import jpype
    ts = jpype.JPackage("edu").cmu.tetrad.search
    s = ts.Ccd(_fisher_z(X, names, alpha, missing))
    s.setVerbose(False)
    return _from_tetrad_graph(s.search(), names, "ccd-pag")


def run_fci(X, names, alpha=0.05, missing="listwise", **kw) -> Estimate:
    start_jvm()
    import jpype
    ts = jpype.JPackage("edu").cmu.tetrad.search
    s = ts.Fci(_fisher_z(X, names, alpha, missing))
    s.setVerbose(False)
    return _from_tetrad_graph(s.search(), names, "pag")


def run_lingd(X, names, b_thresh=0.1, w_thresh=0.1, **kw) -> Estimate:
    """Tetrad の LiNG-D. 完全観測を前提にするので行ごと削除して渡す.

    fit は安定かどうかを問わず候補の B を全部返す. 自作版 (methods.ling_d) と同じく,
    安定な候補のうち辺が最も少ないものを採り, 候補の個数を raw に残す.
    """
    start_jvm()
    import jpype
    ts = jpype.JPackage("edu").cmu.tetrad.search
    s = ts.LingD()
    s.setBThreshold(float(b_thresh))
    s.setWThreshold(float(w_thresh))
    bhats = s.fit(_dataset(_complete_case(X), names))
    mats = []
    for M in bhats:
        A = np.array([[M.get(i, j) for j in range(M.getNumColumns())]
                      for i in range(M.getNumRows())])
        # Tetrad の isStable は固有値分解が収束しないと例外を出すので, 同じ判定
        # (スペクトル半径 < 1) を numpy で行う
        mats.append((A, bool(np.max(np.abs(np.linalg.eigvals(A))) < 1.0)))
    if not mats:
        raise RuntimeError("LiNG-D が候補を返さなかった")
    stable = [A for A, ok in mats if ok]
    cands = stable or [A for A, _ in mats]
    B = min(cands, key=lambda M: int((np.abs(M) > b_thresh).sum()))
    # Tetrad の B は B[i, j] が x_j -> x_i (x = Bx + e). methods.from_weight_matrix と同じ規約
    A = (np.abs(B) > b_thresh).T
    np.fill_diagonal(A, False)
    return Estimate(names, A, A | A.T, kind="cyclic",
                    raw={"n_candidates": len(mats), "n_stable": len(stable), "B": B})


def true_graph(B: np.ndarray, names: list[str], latent: list[str]):
    """係数行列 (B[i, j] は x_j -> x_i) から Tetrad のグラフを作る. latent に挙げた点は未観測."""
    start_jvm()
    import jpype
    tg = jpype.JPackage("edu").cmu.tetrad.graph
    nodes = {}
    lst = jpype.JClass("java.util.ArrayList")()
    for nm in names:
        nd = tg.GraphNode(nm)
        if nm in latent:
            nd.setNodeType(tg.NodeType.LATENT)
        nodes[nm] = nd
        lst.add(nd)
    g = tg.EdgeListGraph(lst)
    d = len(names)
    for i in range(d):
        for j in range(d):
            if i != j and B[i, j] != 0:
                g.addDirectedEdge(nodes[names[j]], nodes[names[i]])
    return g


def run_oracle(alg: str, B: np.ndarray, names: list[str], latent: list[str]) -> dict:
    """d分離の神託で CCD / FCI を回す. 統計検定の誤りを除いた, 手法そのものの出力を見るため.

    巡回グラフの d分離は線形 SCM の条件付き独立と一致する (Spirtes 1995).
    戻り値は辺の文字列と, CCD が巡回の情報を載せる下線つきの三つ組.
    """
    start_jvm()
    import jpype
    ts = jpype.JPackage("edu").cmu.tetrad.search
    tt = jpype.JPackage("edu").cmu.tetrad.search.test
    g = true_graph(B, names, latent)
    test = tt.MsepTest(g)
    s = ts.Ccd(test) if alg == "ccd" else ts.Fci(test)
    out = s.search()
    return {"edges": sorted(str(e) for e in out.getEdges()),
            "underlines": sorted(str(t) for t in out.getUnderLines()),
            "dotted_underlines": sorted(str(t) for t in out.getDottedUnderlines())}


OFFICIAL_TETRAD = {
    "ccd_tetrad": run_ccd,
    "ccd_tetrad_tw": lambda X, names, **kw: run_ccd(X, names, missing="testwise", **kw),
    "fci_tetrad": run_fci,
    "fci_tetrad_tw": lambda X, names, **kw: run_fci(X, names, missing="testwise", **kw),
    "lingd_tetrad": run_lingd,
}

### 2.6 公式実装のラッパ: MVPC (R)・MissDAG・MissNODAG (official_missing)

In [ ]:
%%module official_missing
"""欠測を扱う3手法を, 論文が示す著者公式リポジトリのコードで呼ぶラッパ.

  MVPC      Tu ら (AISTATS 2019)   https://github.com/TURuibo/MVPC        (R, pcalg 上の実装)
  MissDAG   Gao ら (NeurIPS 2022)  https://github.com/ErdunGAO/MissDAG     (Python)
  MissNODAG Sethuraman ら (TMLR 2026) https://github.com/muralikgs/missnodag (Python, torch)

MVPC と MissNODAG の URL は論文本文に書かれている (MVPC は arXiv 版の脚注1,
MissNODAG は本文の脚注1). MissDAG の論文本文には自身のコードの URL が無く,
チェックリストで補足資料に含めたとしている. 上の URL は第一著者のアカウントにある公開版である.

9月の環境 (methods.py) では MVPC を causal-learn の pc(mvpc=True) で回していた.
causal-learn の MVPC は公式実装の移植だが, 同じ結果になるかは確かめていなかったので,
ここで公式実装を呼べるようにして突き合わせる (compare_official.py).

いずれも取得したコードは official/ の下に置き, 中身には手を入れていない.
MissDAG だけは, 線形ガウス版が使う NOTEARS と EM の部分が tensorflow を要しないのに,
パッケージの __init__ と utils.utils が tensorflow を import するので,
必要なモジュールをファイルから直接読み込み, utils.utils には tensorflow の空の代役を渡す
(set_seed 以外では使われていない).
"""
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
import tempfile
import types

import numpy as np

from metrics import Estimate

ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
OFFICIAL = os.path.join(ROOT, "official")
R_SCRIPT = os.path.join(ROOT, "src", "r", "run_mvpc_official.R")
MISSNODAG_PY = os.path.join(OFFICIAL, "missnodag", ".venv", "bin", "python")
MISSNODAG_RUNNER = os.path.join(ROOT, "src", "runners", "missnodag_runner.py")


# ---------------------------------------------------------------------------
# MVPC (R)
# ---------------------------------------------------------------------------
def _from_pcalg_amat(A: np.ndarray, names: list[str], kind: str, raw=None) -> Estimate:
    """pcalg の amat (A[i, j] = 1 は i -> j の辺. 無向辺は両方 1) を共通表現にする."""
    A = np.asarray(A).astype(bool)
    adjacent = A | A.T
    directed = A & ~A.T
    return Estimate(names, directed, adjacent, kind=kind, raw=raw)


def run_mvpc_r(X, names, alpha=0.05, method="permc", seed=0, **kw) -> Estimate:
    with tempfile.TemporaryDirectory() as td:
        inp, out = os.path.join(td, "x.csv"), os.path.join(td, "o.json")
        header = ",".join(names)
        np.savetxt(inp, X, delimiter=",", header=header, comments="", fmt="%.10g")
        # np.savetxt は NaN を "nan" と書く. R の read.csv は "nan" を NA と読まないので直す
        with open(inp) as f:
            txt = f.read().replace("nan", "NA")
        with open(inp, "w") as f:
            f.write(txt)
        r = subprocess.run(["Rscript", R_SCRIPT, inp, out, str(alpha), method, str(seed)],
                           capture_output=True, text=True, timeout=1800)
        if r.returncode != 0 or not os.path.exists(out):
            raise RuntimeError("MVPC(R) が失敗した: " + r.stderr[-800:])
        res = json.load(open(out))
    return _from_pcalg_amat(np.array(res["amat"]), names, "cpdag",
                            raw={"prt_m": res.get("prt_m"), "seconds": res["seconds"]})


# ---------------------------------------------------------------------------
# MissDAG (Python, 線形ガウス版)
# ---------------------------------------------------------------------------
_MISSDAG = None


def _load_missdag():
    global _MISSDAG
    if _MISSDAG is not None:
        return _MISSDAG
    repo = os.path.join(OFFICIAL, "MissDAG")

    def load(modname, relpath):
        spec = importlib.util.spec_from_file_location(modname, os.path.join(repo, relpath))
        mod = importlib.util.module_from_spec(spec)
        sys.modules[modname] = mod
        spec.loader.exec_module(mod)
        return mod

    if "tensorflow" not in sys.modules:
        sys.modules["tensorflow"] = types.ModuleType("tensorflow")  # set_seed 専用の代役
    em_utils = load("miss_methods.em_utils", "miss_methods/em_utils.py")
    sys.modules.setdefault("miss_methods", types.ModuleType("miss_methods")).em_utils = em_utils
    mdg = load("missdag_gaussian", "miss_methods/miss_dag_gaussian.py")
    nt = load("missdag_notears", "dag_methods/notears/notears.py")
    ut = load("missdag_utils", "utils/utils.py")
    _MISSDAG = (mdg.miss_dag_gaussian, nt.Notears, ut.postprocess)
    return _MISSDAG


def run_missdag(X, names, lambda_1=2e-2, em_iter=10, graph_thres=0.3,
                equal_variances=False, seed=0, **kw) -> Estimate:
    """MissDAG + NOTEARS (線形ガウス). 既定値は公式 utils/config.py のもの.

    標準化したデータでは誤差分散がそろわないので, equal_variances は既定で False にする.
    MissDAG の B は B[i, j] が i -> j (X @ B). 出力は DAG.
    """
    miss_dag_gaussian, Notears, postprocess = _load_missdag()
    np.random.seed(seed)
    mask = ~np.isnan(X)
    B_est, _, _ = miss_dag_gaussian(X, mask, Notears(lambda_1), em_iter, equal_variances)
    _, B_bin = postprocess(B_est, graph_thres)
    A = B_bin.astype(bool)
    np.fill_diagonal(A, False)
    return Estimate(names, A, A | A.T, kind="dag", raw={"B": B_est})


# ---------------------------------------------------------------------------
# MissNODAG (Python, 専用の venv で subprocess)
# ---------------------------------------------------------------------------
def run_missnodag(X, names, max_epochs=100, seed=0, adj_threshold=None, **kw) -> Estimate:
    if not os.path.exists(MISSNODAG_PY):
        raise RuntimeError("official/missnodag/.venv が無い. official/missnodag で uv sync する")
    with tempfile.TemporaryDirectory() as td:
        inp, out = os.path.join(td, "x.npz"), os.path.join(td, "o.json")
        np.savez(inp, X=X)
        r = subprocess.run([MISSNODAG_PY, MISSNODAG_RUNNER, inp, out, str(max_epochs), str(seed)],
                           capture_output=True, text=True, timeout=7200)
        if r.returncode != 0 or not os.path.exists(out):
            raise RuntimeError("MissNODAG が失敗した: " + r.stderr[-800:])
        res = json.load(open(out))
    if res.get("error"):
        raise RuntimeError("MissNODAG の学習が失敗した: " + res["error"])
    W = np.array(res["w_adj"])
    thr = res["adj_threshold"] if adj_threshold is None else adj_threshold
    A = W > thr
    np.fill_diagonal(A, False)
    return Estimate(names, A, A | A.T, kind="cyclic",
                    raw={"w_adj": W, "seconds": res["seconds"]})


OFFICIAL_MISSING = {
    "mvpc_r_td": lambda X, names, **kw: run_mvpc_r(X, names, method="td", **kw),
    "mvpc_r_permc": lambda X, names, **kw: run_mvpc_r(X, names, method="permc", **kw),
    "mvpc_r_drw": lambda X, names, **kw: run_mvpc_r(X, names, method="drw", **kw),
    "missdag": run_missdag,
    "missnodag": run_missnodag,
}

### 2.7 生成器の診断 (diagnostics)

In [ ]:
%%module diagnostics
"""生成器が仕様どおりかを検査する診断.

検証環境を作ったあと最初にやるべきなのは, 生成器が「巡回レベル」と「欠測機構」を
本当に独立に制御できているかを数値で確かめることである. ここが崩れていると,
9月以降の「既存手法の破綻条件」の実験結果がすべて生成器の副作用と区別できなくなる.

検査項目
  D1 rho の制御     : 目標スペクトル半径に実測が一致するか. ループ利得と作用素ノルムの挙動
  D2 平衡           : 閉形式 (I-B)^{-1} e と反復の一致, 理論共分散と標本共分散の一致
  D3 安定性の境界   : rho >= 1 で反復が発散すること (評価計画の rho 上限の根拠)
  D4 欠測率の較正   : 目標欠測率と実測欠測率の一致 (5〜60%)
  D5 機構の同定     : 入れた機構が MCAR/MAR/MNAR として実際に区別できるか
  D6 巡回の識別可能性: 2-cycle の条件付き独立パターンが非巡回のそれと一致しないか
  D7 完全ケース分析 : 自己マスキング下で行ごと削除が相関構造を歪める量

実行:  uv run python src/diagnostics.py
"""
from __future__ import annotations

import itertools
import json
import os
import sys

import numpy as np

from generator import (MissingSpec, Scenario, build_B, draw_noise, generate,
                       loop_gain, sample_equilibrium, sample_iterate,
                       spectral_radius, variables)

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
os.makedirs(OUT, exist_ok=True)


# ---------------------------------------------------------------------------
# 補助
# ---------------------------------------------------------------------------
def population_cov(B: np.ndarray, noise_scale) -> np.ndarray:
    """平衡分布の理論共分散 (I-B)^{-1} Sigma_e (I-B)^{-T}.

    noise_scale はスカラでも, 変数ごとの標準偏差ベクトルでもよい.
    """
    d = B.shape[0]
    A = np.linalg.inv(np.eye(d) - B)
    sd = np.full(d, float(noise_scale)) if np.isscalar(noise_scale) else np.asarray(noise_scale)
    return A @ np.diag(sd ** 2) @ A.T


def partial_corr(S: np.ndarray, i: int, j: int, cond: tuple[int, ...]) -> float:
    """共分散行列 S から偏相関 rho(i,j | cond) を計算する."""
    sel = [i, j, *cond]
    sub = S[np.ix_(sel, sel)]
    P = np.linalg.pinv(sub)
    den = np.sqrt(P[0, 0] * P[1, 1])
    if den <= 0:
        return 0.0
    return float(-P[0, 1] / den)


def ci_signature(S: np.ndarray, names: list[str], tol: float = 1e-8) -> set[str]:
    """理論共分散が含意する条件付き独立の一覧 (線形ガウスなので偏相関=0 と同値)."""
    d = len(names)
    sig = set()
    for i, j in itertools.combinations(range(d), 2):
        rest = [k for k in range(d) if k not in (i, j)]
        for r in range(len(rest) + 1):
            for cond in itertools.combinations(rest, r):
                if abs(partial_corr(S, i, j, cond)) < tol:
                    c = ",".join(names[k] for k in cond) or "{}"
                    sig.add(f"{names[i]} _||_ {names[j]} | {c}")
    return sig


def submatrix_by_names(S: np.ndarray, names: list[str], keep: list[str]) -> tuple[np.ndarray, list[str]]:
    idx = [names.index(v) for v in keep]
    return S[np.ix_(idx, idx)], keep


# ---------------------------------------------------------------------------
# D1 rho の制御
# ---------------------------------------------------------------------------
def d1_rho_control() -> list[dict]:
    rows = []
    for cycle in ("acyclic2", "acyclic3", "2-cycle", "3-cycle"):
        for rho in (0.1, 0.3, 0.5, 0.7, 0.9):
            sc = Scenario(cycle=cycle, rho=rho)
            B, names, cyc = build_B(sc)
            idx = {v: i for i, v in enumerate(names)}
            rows.append({
                "cycle": cycle,
                "rho_target": 0.0 if cycle == "acyclic" else rho,
                "rho_actual": spectral_radius(B),
                "loop_gain": loop_gain(B, [idx[v] for v in cyc]),
                "op_norm": float(np.linalg.norm(B, 2)),
            })
    return rows


# ---------------------------------------------------------------------------
# D2 平衡
# ---------------------------------------------------------------------------
def d2_equilibrium(n: int = 20000) -> list[dict]:
    rows = []
    for cycle in ("acyclic2", "acyclic3", "2-cycle", "3-cycle"):
        for rho in (0.3, 0.6, 0.9):
            sc = Scenario(cycle=cycle, rho=rho, seed=1)
            B, names, _ = build_B(sc)
            rng = np.random.default_rng(1)
            E = draw_noise(sc.noise, sc.noise_scale, (n, len(names)), rng)
            x_closed = sample_equilibrium(B, E)
            x_iter, iters, conv = sample_iterate(B, E)
            S_emp = np.cov(x_closed, rowvar=False)
            S_pop = population_cov(B, sc.noise_scale)
            rows.append({
                "cycle": cycle,
                "rho": 0.0 if cycle == "acyclic" else rho,
                "iter_converged": bool(conv),
                "iters": iters,
                "max_abs_diff_closed_vs_iter": float(np.max(np.abs(x_closed - x_iter))),
                "max_rel_diff_cov": float(np.max(np.abs(S_emp - S_pop)) / np.max(np.abs(S_pop))),
            })
    return rows


# ---------------------------------------------------------------------------
# D3 安定性の境界
# ---------------------------------------------------------------------------
def d3_stability(n: int = 2000) -> list[dict]:
    rows = []
    for rho in (0.5, 0.9, 0.95, 0.99, 1.0, 1.05, 1.2):
        sc = Scenario(cycle="2-cycle", rho=rho, seed=2)
        B, names, _ = build_B(sc)
        rng = np.random.default_rng(2)
        E = draw_noise(sc.noise, sc.noise_scale, (n, len(names)), rng)
        x_iter, iters, conv = sample_iterate(B, E, n_iter=2000)
        det = float(np.linalg.det(np.eye(len(names)) - B))
        # rho=1 かつループ利得が正のとき det(I-B) = 1 - rho**k = 0 となり平衡が一意でない.
        # 閉形式が定義できない領域があることを, 例外ではなく値として記録する.
        defined = abs(det) > 1e-10
        pop_var = float(population_cov(B, sc.noise_scale)[2, 2]) if defined else None
        rows.append({
            "rho": rho,
            "iter_converged": bool(conv),
            "iters_to_converge": iters if conv else None,
            "max_abs_x_iter": float(np.max(np.abs(x_iter))) if np.all(np.isfinite(x_iter)) else float("inf"),
            "det_I_minus_B": det,
            "pop_var_X1": pop_var,
            "closed_form_defined": defined,
        })
    return rows


# ---------------------------------------------------------------------------
# D4 欠測率の較正
# ---------------------------------------------------------------------------
def d4_missing_rate(n: int = 20000) -> list[dict]:
    rows = []
    for mech in ("mcar", "mar", "mnar_self"):
        for rate in (0.05, 0.15, 0.30, 0.45, 0.60):
            for target, where in (("Y", "巡回外"), ("X1", "巡回内")):
                spec = MissingSpec(target=target, mechanism=mech, rate=rate,
                                   driver="Z" if mech == "mar" else None)
                sc = Scenario(cycle="2-cycle", rho=0.6, missing=[spec], seed=3)
                out = generate(sc, n)
                info = out["meta"]["missing"][target]
                rows.append({
                    "mechanism": mech, "target": target, "where": where,
                    "rate_target": rate, "rate_realized": info["realized_rate"],
                    "abs_error": abs(info["realized_rate"] - rate),
                })
    return rows


# ---------------------------------------------------------------------------
# D5 機構の同定
# ---------------------------------------------------------------------------
def d5_mechanism(n: int = 40000) -> list[dict]:
    """入れた機構が実際に MCAR/MAR/MNAR として区別できるかを, 完全データ側から測る.

    corr(M_t, t)      : 欠測指標と当該変数の値の相関. MNAR 自己マスキングで大きくなる
    pcorr(M_t, t | Z) : 駆動変数 Z を条件付けた後に残る自分の値への依存.
                        MAR ならここが 0 に近づき, 自己マスキングでは残る
    corr(M_t, Z)      : 駆動変数への依存
    """
    rows = []
    for mech in ("mcar", "mar", "mnar_self"):
        for target in ("Y", "X1"):
            spec = MissingSpec(target=target, mechanism=mech, rate=0.3,
                               driver="Z" if mech == "mar" else None)
            sc = Scenario(cycle="2-cycle", rho=0.6, missing=[spec], seed=4)
            out = generate(sc, n)
            names = out["meta"]["names"]
            idx = {v: i for i, v in enumerate(names)}
            X = out["X_full"]
            m = out["M"][:, idx[target]].astype(float)
            D = np.column_stack([m, X[:, idx[target]], X[:, idx["Z"]]])
            S = np.cov(D, rowvar=False)
            rows.append({
                "mechanism": mech, "target": target,
                "corr_M_target": float(S[0, 1] / np.sqrt(S[0, 0] * S[1, 1])),
                "corr_M_Z": float(S[0, 2] / np.sqrt(S[0, 0] * S[2, 2])),
                "pcorr_M_target_given_Z": partial_corr(S, 0, 1, (2,)),
                "pcorr_M_Z_given_target": partial_corr(S, 0, 2, (1,)),
            })
    return rows


# ---------------------------------------------------------------------------
# D6 巡回の識別可能性 (条件付き独立のパターン)
# ---------------------------------------------------------------------------
def d6_ci_signature() -> dict:
    """理論共分散が含意する条件付き独立を, 非巡回 / 2-cycle / 長さ3 で比較する.

    制約ベース手法が使える情報は条件付き独立だけである. したがって, 2-cycle の
    条件付き独立パターンが非巡回のものと一致するなら, 観測データだけを使う限り
    どんな制約ベース手法でもこの2つを区別できない. これは提案手法の識別可能性の
    上限を決める事実なので, 検証環境の最初の出力として確認しておく.
    """
    res = {}
    for cycle in ("acyclic2", "acyclic3", "2-cycle", "3-cycle"):
        for rho in (0.3, 0.6):
            for latent in (False, True):
                sc = Scenario(cycle=cycle, rho=rho, latent=latent)
                B, names, _ = build_B(sc)
                S = population_cov(B, sc.noise_scale)
                keep = [v for v in names if not (latent and v == "L")]
                Sk, nk = submatrix_by_names(S, names, keep)
                key = f"{cycle}|rho={0.0 if cycle in ('acyclic2', 'acyclic3') else rho}|latent={latent}"
                res[key] = sorted(ci_signature(Sk, nk))
    # 変数の本数をそろえた非巡回の対照と比べ, 閉じる辺の有無だけが差になるようにする
    verdict = {}
    for rho in (0.3, 0.6):
        for latent in (False, True):
            for cycle, ctrl in (("2-cycle", "acyclic2"), ("3-cycle", "acyclic3")):
                a = set(res[f"{ctrl}|rho=0.0|latent={latent}"])
                b = set(res[f"{cycle}|rho={rho}|latent={latent}"])
                verdict[f"{ctrl} vs {cycle}|rho={rho}|latent={latent}"] = {
                    "same_ci_set": a == b,
                    "only_in_acyclic": sorted(a - b),
                    "only_in_cyclic": sorted(b - a),
                }
    return {"signatures": res, "comparison": verdict}


# ---------------------------------------------------------------------------
# D6b 条件付き独立の一致が係数の特殊値によるものでないかの確認
# ---------------------------------------------------------------------------
def d6b_random_coefficients(n_draw: int = 200) -> dict:
    """構造係数と誤差分散を無作為に振っても, 巡回と非巡回の条件付き独立が一致するか.

    D6 は係数を固定した1点での結果なので, たまたま偏相関が消えた可能性を潰しておく.
    巡回の辺は rho に固定したまま, それ以外の辺の係数と各変数の誤差分散を毎回引き直す.
    """
    rng = np.random.default_rng(7)
    result = {}
    for cycle, ctrl in (("2-cycle", "acyclic2"), ("3-cycle", "acyclic3")):
        for latent in (False, True):
            same = 0
            examples = []
            for t in range(n_draw):
                rho = float(rng.uniform(0.15, 0.9))
                sigs = {}
                # 同じ乱数列から同じ「巡回以外の係数」を作るため, 引いた値を共有する
                coef_seed = int(rng.integers(0, 2 ** 31 - 1))
                for kind in (ctrl, cycle):
                    sc = Scenario(cycle=kind, rho=rho, latent=latent)
                    B, names, cyc = build_B(sc)
                    r2 = np.random.default_rng(coef_seed)
                    cyc_pairs = set()
                    k = sc.cycle_len
                    idx = {v: i for i, v in enumerate(names)}
                    if k:
                        for a in range(k):
                            cyc_pairs.add((idx[cyc[(a + 1) % k]], idx[cyc[a]]))
                    for i in range(len(names)):
                        for j in range(len(names)):
                            if B[i, j] != 0 and (i, j) not in cyc_pairs:
                                B[i, j] = r2.uniform(0.3, 1.2) * r2.choice([-1.0, 1.0])
                    sd = r2.uniform(0.5, 2.0, size=len(names))
                    S = population_cov(B, sd)
                    keep = [v for v in names if not (latent and v == "L")]
                    Sk, nk = submatrix_by_names(S, names, keep)
                    sigs[kind] = ci_signature(Sk, nk, tol=1e-7)
                if sigs[ctrl] == sigs[cycle]:
                    same += 1
                elif len(examples) < 2:
                    examples.append({
                        "rho": rho,
                        "only_in_acyclic": sorted(sigs[ctrl] - sigs[cycle]),
                        "only_in_cyclic": sorted(sigs[cycle] - sigs[ctrl]),
                    })
            result[f"{ctrl} vs {cycle}|latent={latent}"] = {
                "draws": n_draw, "same_ci_set": same,
                "same_ratio": same / n_draw, "examples_of_difference": examples,
            }
    return result


# ---------------------------------------------------------------------------
# D6c 2-cycle が条件付き独立で見えるための周辺構造の条件
# ---------------------------------------------------------------------------
def _ci_equal(edges_cyclic, edges_acyclic, observed, allv, n_draw, rng) -> tuple[int, int]:
    """2つの辺集合が, 観測変数上で同じ条件付き独立を含意する回数を数える."""
    idx = {v: i for i, v in enumerate(allv)}
    same = 0
    for _ in range(n_draw):
        rho = float(rng.uniform(0.2, 0.9))
        coef = {}
        for e in set(edges_cyclic) | set(edges_acyclic):
            coef[e] = float(rng.uniform(0.3, 1.2)) * float(rng.choice([-1.0, 1.0]))
        sd = rng.uniform(0.5, 2.0, size=len(allv))
        sigs = []
        for edges, is_cyc in ((edges_acyclic, False), (edges_cyclic, True)):
            B = np.zeros((len(allv), len(allv)))
            for (a, b) in edges:
                w = coef[(a, b)]
                if is_cyc and {a, b} == {"X1", "X2"}:
                    w = rho          # 巡回の2辺は rho に固定する
                B[idx[b], idx[a]] = w
            S = population_cov(B, sd)
            sel = [idx[v] for v in observed]
            sigs.append(ci_signature(S[np.ix_(sel, sel)], list(observed), tol=1e-7))
        if sigs[0] == sigs[1]:
            same += 1
    return same, n_draw


def d6c_identifiability_map(n_draw: int = 60) -> list[dict]:
    """2-cycle X1 <-> X2 が, 周辺構造を変えたときに条件付き独立で区別できるかの地図.

    D6b で, 図1 の構造 (Z -> X1, 未観測 L -> X1,X2, X1,X2 -> Y) では
    2-cycle と非巡回が観測変数上で同じ条件付き独立を含意することが分かった.
    どの辺がそれを決めているのかを, 辺の有無を振って切り分ける.
    区別できる = 制約ベース手法に情報がある, ということなので, 提案手法が
    どの範囲を主張できるかの上限をここで決められる.
    """
    allv = ["Z", "L", "X1", "X2", "Y"]
    observed = ["Z", "X1", "X2", "Y"]
    rng = np.random.default_rng(21)
    rows = []
    for z_edge in (None, ("Z", "X1"), ("Z", "X2")):
        for l_mode in ("none", "both", "X1のみ", "X2のみ"):
            for y_mode in ("both", "X1のみ", "なし"):
                base = []
                if z_edge:
                    base.append(z_edge)
                if l_mode == "both":
                    base += [("L", "X1"), ("L", "X2")]
                elif l_mode == "X1のみ":
                    base.append(("L", "X1"))
                elif l_mode == "X2のみ":
                    base.append(("L", "X2"))
                if y_mode == "both":
                    base += [("X1", "Y"), ("X2", "Y")]
                elif y_mode == "X1のみ":
                    base.append(("X1", "Y"))
                cyc = [*base, ("X1", "X2"), ("X2", "X1")]
                # 非巡回の対照は両方の向きを取る. 片方だけと比べると, 対照側の
                # 非対称性を巡回の検出力と取り違える (Z->X2 の行で実際に起きた)
                s12, tot = _ci_equal(cyc, [*base, ("X1", "X2")], observed, allv, n_draw, rng)
                s21, _ = _ci_equal(cyc, [*base, ("X2", "X1")], observed, allv, n_draw, rng)
                rows.append({
                    "Zの辺": "なし" if z_edge is None else f"Z->{z_edge[1]}",
                    "未観測交絡L": l_mode,
                    "Yへの辺": y_mode,
                    "同一CI率_X1toX2": s12 / tot,
                    "同一CI率_X2toX1": s21 / tot,
                    "区別できる": s12 == 0 and s21 == 0,
                })
    return rows


# ---------------------------------------------------------------------------
# D7 完全ケース分析の歪み
# ---------------------------------------------------------------------------
def d7_complete_case(n: int = 40000) -> list[dict]:
    """自己マスキング下で行ごと削除 (listwise deletion) が相関をどれだけ歪めるか."""
    rows = []
    for mech in ("mcar", "mar", "mnar_self"):
        for rate in (0.15, 0.30, 0.60):
            spec = MissingSpec(target="Y", mechanism=mech, rate=rate,
                               driver="Z" if mech == "mar" else None)
            sc = Scenario(cycle="2-cycle", rho=0.6, missing=[spec], seed=5)
            out = generate(sc, n)
            names = out["meta"]["names"]
            idx = {v: i for i, v in enumerate(names)}
            X = out["X_full"]
            keep = ~out["M"][:, idx["Y"]]
            full = np.corrcoef(X, rowvar=False)
            cc = np.corrcoef(X[keep], rowvar=False)
            pairs = [("X1", "Y"), ("X2", "Y"), ("X1", "X2"), ("Z", "Y")]
            rec = {"mechanism": mech, "rate": rate, "n_complete": int(keep.sum())}
            for a, b in pairs:
                rec[f"corr_{a}_{b}_full"] = float(full[idx[a], idx[b]])
                rec[f"corr_{a}_{b}_cc"] = float(cc[idx[a], idx[b]])
                rec[f"bias_{a}_{b}"] = float(cc[idx[a], idx[b]] - full[idx[a], idx[b]])
            rows.append(rec)
    return rows


# ---------------------------------------------------------------------------
def main() -> None:
    result = {
        "D1_rho_control": d1_rho_control(),
        "D2_equilibrium": d2_equilibrium(),
        "D3_stability": d3_stability(),
        "D4_missing_rate": d4_missing_rate(),
        "D5_mechanism": d5_mechanism(),
        "D6_ci_signature": d6_ci_signature(),
        "D6b_random_coefficients": d6b_random_coefficients(),
        "D6c_identifiability_map": d6c_identifiability_map(),
        "D7_complete_case": d7_complete_case(),
    }
    path = os.path.join(OUT, "diagnostics.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=1)
    print(f"書き出し: {path}")

    print("\n[D1] rho の制御")
    print(f"{'巡回':<9}{'目標rho':>8}{'実測rho':>9}{'ループ利得':>11}{'作用素ノルム':>13}")
    for r in result["D1_rho_control"]:
        print(f"{r['cycle']:<9}{r['rho_target']:>8.2f}{r['rho_actual']:>9.4f}"
              f"{r['loop_gain']:>11.4f}{r['op_norm']:>13.4f}")

    print("\n[D2] 平衡 (閉形式と反復, 理論共分散と標本共分散)")
    for r in result["D2_equilibrium"]:
        print(f"  {r['cycle']:<9} rho={r['rho']:.1f} 収束={r['iter_converged']} "
              f"反復={r['iters']:>4} 最大差={r['max_abs_diff_closed_vs_iter']:.2e} "
              f"共分散相対差={r['max_rel_diff_cov']:.4f}")

    print("\n[D3] 安定性の境界")
    for r in result["D3_stability"]:
        pv = "定義不能" if r["pop_var_X1"] is None else f"{r['pop_var_X1']:.3f}"
        print(f"  rho={r['rho']:<5} 反復収束={str(r['iter_converged']):<6}"
              f"反復回数={str(r['iters_to_converge']):<6} |x|max={r['max_abs_x_iter']:.3e} "
              f"det(I-B)={r['det_I_minus_B']:>8.4f} 理論分散(X1)={pv}")

    print("\n[D4] 欠測率の較正 (目標と実測の最大絶対誤差)")
    err = max(r["abs_error"] for r in result["D4_missing_rate"])
    print(f"  最大絶対誤差 = {err:.4f} ({len(result['D4_missing_rate'])} 条件)")

    print("\n[D5] 機構の同定")
    print(f"{'機構':<11}{'対象':<5}{'corr(M,t)':>11}{'corr(M,Z)':>11}"
          f"{'pcorr(M,t|Z)':>14}{'pcorr(M,Z|t)':>14}")
    for r in result["D5_mechanism"]:
        print(f"{r['mechanism']:<11}{r['target']:<5}{r['corr_M_target']:>11.4f}"
              f"{r['corr_M_Z']:>11.4f}{r['pcorr_M_target_given_Z']:>14.4f}"
              f"{r['pcorr_M_Z_given_target']:>14.4f}")

    print("\n[D6] 条件付き独立のパターン (非巡回と巡回が区別できるか)")
    for k, v in result["D6_ci_signature"]["comparison"].items():
        print(f"  {k}: 同一={v['same_ci_set']}")
        if not v["same_ci_set"]:
            if v["only_in_acyclic"]:
                print(f"      非巡回のみ: {v['only_in_acyclic']}")
            if v["only_in_cyclic"]:
                print(f"      巡回のみ  : {v['only_in_cyclic']}")

    print("\n[D6b] 係数と誤差分散を無作為に振ったときの一致率")
    for k, v in result["D6b_random_coefficients"].items():
        print(f"  {k}: 一致 {v['same_ci_set']}/{v['draws']} ({v['same_ratio']:.3f})")
        for ex in v["examples_of_difference"]:
            print(f"      不一致例 rho={ex['rho']:.3f} 非巡回のみ={ex['only_in_acyclic']} "
                  f"巡回のみ={ex['only_in_cyclic']}")

    print("\n[D6c] 2-cycle が条件付き独立で区別できる周辺構造")
    print(f"{'Zの辺':<10}{'未観測交絡L':<12}{'Yへの辺':<9}{'X1->X2と同一':>13}"
          f"{'X2->X1と同一':>13}  判定")
    for r in result["D6c_identifiability_map"]:
        v = "区別できる" if r["区別できる"] else "区別できない"
        print(f"{r['Zの辺']:<10}{r['未観測交絡L']:<12}{r['Yへの辺']:<9}"
              f"{r['同一CI率_X1toX2']:>13.2f}{r['同一CI率_X2toX1']:>13.2f}  {v}")

    print("\n[D7] 完全ケース分析の歪み (相関のずれ)")
    print(f"{'機構':<11}{'欠測率':>7}{'X1-Y':>9}{'X2-Y':>9}{'X1-X2':>9}{'Z-Y':>9}")
    for r in result["D7_complete_case"]:
        print(f"{r['mechanism']:<11}{r['rate']:>7.2f}{r['bias_X1_Y']:>9.4f}"
              f"{r['bias_X2_Y']:>9.4f}{r['bias_X1_X2']:>9.4f}{r['bias_Z_Y']:>9.4f}")


if __name__ == "__main__":
    main()

### 2.8 4軸の実験ランナー (run_experiment)

中間発表の評価4軸を基準条件から1軸ずつ振る。--official で公式実装を加え、--only=手法,... で手法を絞る。

In [ ]:
%%module run_experiment
"""既存手法の破綻条件を測る実験.

中間発表スライド7の4軸を, 基準条件から1軸ずつ振る形 (one-factor-at-a-time) で回す.
全直積にしないのは, 軸間の交互作用より先に「どの軸のどこで壊れるか」を確定させたいためと,
交互作用は破綻点が決まってから必要な組み合わせだけ見ればよいためである.

基準条件は図1の構造に合わせる:
  2-cycle, rho=0.6, 未観測交絡あり, ガウス誤差, 開催成果 Y に自己マスキング 30%.

「破綻」の操作的定義:
  パラメータを1つしか持たない自明なベースライン (corr_thresh) の骨格 F1 を下回ること.
  手法ごとの調整の巧拙に左右されにくい判定にするため, 絶対値の閾値は使わない.

実行:  uv run python src/run_experiment.py            (既定: 5 シード, n=2000)
       uv run python src/run_experiment.py 10 5000    (シード数, サンプル数)
       uv run python src/run_experiment.py 5 2000 --official
           論文が示す公式実装 (Tetrad の CCD・FCI・LiNG-D, 公式 R の MVPC, MissDAG, MissNODAG)
           を加えて回す. 出力は outputs/experiment_official.csv
"""
from __future__ import annotations

import csv
import os
import sys
import time

import numpy as np

from generator import MissingSpec, Scenario, generate
from methods import ASSUMPTIONS, METHODS, run_method, run_mvpc_fixed
from metrics import score_all, true_structure

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
os.makedirs(OUT, exist_ok=True)

# 標準化は既定で有効にする. 無効だと varsortability が 1.000 になり,
# 分散の並びだけで因果順序が当たってしまう (ICA-LiNGAM の向き付け正解率が
# ガウス誤差で 0.856 と出るが, 標準化すると 0.331 に落ちる. 前者は人工物).
BASE = {"cycle": "2-cycle", "rho": 0.6, "latent": True, "noise": "gaussian",
        "mech": "mnar_self", "rate": 0.30, "target": "Y", "n_extra": 6,
        "standardize": True}

AXES = {
    "巡回の複雑さ": ("cycle", ["acyclic2", "2-cycle", "acyclic3", "3-cycle"]),
    "フィードバック利得": ("rho", [0.1, 0.3, 0.5, 0.7, 0.9]),
    "欠測機構": ("mech", ["none", "mcar", "mar", "mnar_self"]),
    "欠測率": ("rate", [0.05, 0.15, 0.30, 0.45, 0.60]),
    "欠測の位置": ("target", ["Y", "X1"]),
    "未観測交絡": ("latent", [False, True]),
    "ノイズ分布": ("noise", ["gaussian", "uniform", "laplace"]),
    "変数の本数": ("n_extra", [0, 6, 12]),
    "標準化": ("standardize", [False, True]),
}

BASELINE = "corr_thresh"


def make_scenario(cfg: dict, seed: int) -> Scenario:
    missing = []
    if cfg["mech"] != "none":
        missing = [MissingSpec(target=cfg["target"], mechanism=cfg["mech"],
                               rate=cfg["rate"],
                               driver="Z" if cfg["mech"] == "mar" else None)]
    return Scenario(cycle=cfg["cycle"], rho=cfg["rho"], latent=cfg["latent"],
                    noise=cfg["noise"], missing=missing, n_extra=cfg["n_extra"],
                    standardize=cfg["standardize"], seed=seed)


def run_one(cfg: dict, seed: int, n: int) -> list[dict]:
    sc = make_scenario(cfg, seed)
    out = generate(sc, n)
    X, names = out["X_obs"], out["obs_names"]
    truth = true_structure(out["B_obs"], names)
    meta = out["meta"]
    rows = []
    for m in METHODS:
        t0 = time.time()
        est, err = run_method(m, X, names)
        rec = {"method": m, "seed": seed, "n": n, "error": err or "",
               "seconds": round(time.time() - t0, 3),
               "rho_actual": meta["rho_actual"],
               "var_X1": meta["marginal_var"]["X1"],
               "varsortability": meta["varsortability"],
               "realized_rate": (list(meta["missing"].values())[0]["realized_rate"]
                                 if meta["missing"] else 0.0),
               **{k: cfg[k] for k in BASE}}
        if est is not None:
            rec.update({k: (float(v) if isinstance(v, (int, float, np.floating, np.integer))
                            else v) for k, v in score_all(est, truth).items()})
        rows.append(rec)
    return rows


def main() -> None:
    official = "--official" in sys.argv
    argv = [a for a in sys.argv[1:] if not a.startswith("--")]
    n_seeds = int(argv[0]) if len(argv) > 0 else 5
    n = int(argv[1]) if len(argv) > 1 else 2000
    if official:
        from official_missing import OFFICIAL_MISSING
        from official_tetrad import OFFICIAL_TETRAD
        # Tetrad の LiNG-D は対角が非零になる並べ替えを全列挙するので, ガウス誤差で
        # ICA の分離行列が密になると13変数では3分を超える. 自作版との一致は
        # compare_official.py で別に確かめているので, ここでは入れない
        for k in ("ccd_tetrad", "fci_tetrad"):
            METHODS[k] = OFFICIAL_TETRAD[k]
        # MissNODAG は1回あたり数十秒かかり, 全条件では半日を超えるので
        # --with-missnodag を付けたときだけ入れる. 通常は missnodag_axes.py で条件を絞って回す
        keys = ["mvpc_r_permc", "mvpc_r_drw", "missdag"]
        if "--with-missnodag" in sys.argv:
            keys.append("missnodag")
        for k in keys:
            METHODS[k] = OFFICIAL_MISSING[k]
        METHODS["mvpc_fixed"] = run_mvpc_fixed

    conditions = []
    seen = set()
    for axis, (key, levels) in AXES.items():
        for lv in levels:
            cfg = {**BASE, key: lv}
            tag = tuple(sorted(cfg.items(), key=lambda kv: kv[0]))
            conditions.append({"axis": axis, "level": str(lv), "cfg": cfg,
                               "duplicate": tag in seen})
            seen.add(tag)

    only = [a.split("=", 1)[1] for a in sys.argv[1:] if a.startswith("--only=")]
    tag = ""
    if only:
        keep = set(only[0].split(","))
        if "mvpc_fixed" in keep:
            METHODS["mvpc_fixed"] = run_mvpc_fixed
        for k in list(METHODS):
            if k not in keep:
                del METHODS[k]
        tag = "_only"
    path = os.path.join(OUT, ("experiment_official" if official else "experiment") + tag + ".csv")
    rows_all = []
    t_start = time.time()
    for ci, cond in enumerate(conditions, 1):
        for seed in range(n_seeds):
            for r in run_one(cond["cfg"], seed, n):
                r["axis"] = cond["axis"]
                r["level"] = cond["level"]
                rows_all.append(r)
        done = time.time() - t_start
        print(f"[{ci}/{len(conditions)}] {cond['axis']}={cond['level']} "
              f"({done:.0f}s 経過)", flush=True)

    keys = sorted({k for r in rows_all for k in r})
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows_all)
    print(f"\n書き出し: {path}  ({len(rows_all)} 行, {time.time()-t_start:.0f}s)")
    summarize(rows_all)


def summarize(rows: list[dict]) -> None:
    import statistics as st

    def agg(sel, field):
        vals = [r[field] for r in sel if r.get("error") == "" and field in r
                and r[field] is not None and not (isinstance(r[field], float) and np.isnan(r[field]))]
        return st.mean(vals) if vals else float("nan")

    axes = []
    for r in rows:
        if (r["axis"], r["level"]) not in axes:
            axes.append((r["axis"], r["level"]))

    print("\n=== 骨格 F1 (平均) ===")
    methods = list(METHODS)
    head = f"{'軸':<12}{'水準':<10}" + "".join(f"{m[:9]:>10}" for m in methods)
    print(head)
    for axis, level in axes:
        sel_all = [r for r in rows if r["axis"] == axis and r["level"] == level]
        line = f"{axis:<12}{level:<10}"
        for m in methods:
            v = agg([r for r in sel_all if r["method"] == m], "skeleton_f1")
            line += f"{v:>10.3f}" if v == v else f"{'err':>10}"
        print(line)

    print("\n=== 2-cycle の再現率 (真に 2-cycle がある条件のみ) ===")
    for axis, level in axes:
        sel_all = [r for r in rows if r["axis"] == axis and r["level"] == level
                   and r.get("true_2cycles", 0)]
        if not sel_all:
            continue
        line = f"{axis:<12}{level:<10}"
        for m in methods:
            v = agg([r for r in sel_all if r["method"] == m], "cycle_recall")
            line += f"{v:>10.3f}" if v == v else f"{'-':>10}"
        print(line)

    print("\n=== 破綻 (自明なベースラインを同一シードで下回る) ===")
    print("判定: シードごとに差をとり, 平均差 + 1.96*標準誤差 < 0 のとき破綻とする")
    for axis, level in axes:
        sel_all = [r for r in rows if r["axis"] == axis and r["level"] == level]
        broken = []
        for base_name in (BASELINE, "complete_graph"):
            bydseed = {r["seed"]: r.get("skeleton_f1") for r in sel_all
                       if r["method"] == base_name and r.get("error") == ""}
            for m in methods:
                if m in (BASELINE, "complete_graph"):
                    continue
                diffs = [r["skeleton_f1"] - bydseed[r["seed"]] for r in sel_all
                         if r["method"] == m and r.get("error") == ""
                         and r["seed"] in bydseed and r.get("skeleton_f1") is not None]
                if len(diffs) < 2:
                    continue
                mu = st.mean(diffs)
                se = st.stdev(diffs) / np.sqrt(len(diffs)) if len(diffs) > 1 else 0.0
                if mu + 1.96 * se < 0:
                    broken.append(f"{m} vs {base_name} ({mu:+.3f})")
        print(f"{axis:<12}{level:<10} 破綻: {', '.join(broken) if broken else 'なし'}")

    print("\n=== 誤り ===")
    errs = {}
    for r in rows:
        if r.get("error"):
            errs.setdefault((r["method"], r["error"][:70]), 0)
            errs[(r["method"], r["error"][:70])] += 1
    for (m, e), c in sorted(errs.items(), key=lambda kv: -kv[1]):
        print(f"  {m:<15} x{c:<4} {e}")
    if not errs:
        print("  なし")


if __name__ == "__main__":
    main()

## 3. 9月の検証

### 3.1 生成器の診断と、2-cycle の識別可能性

D1〜D7: rho の定義, 平衡の存在, 欠測率の較正, 機構の同定, 条件付き独立の集合の比較 (D6), 周辺構造の地図。

In [ ]:
ARGS = () if QUICK else ()
if RUN["diagnostics"]:
    run_main("diagnostics")

### 3.2 d分離での確認

図1 の設定 (両方の巡回ノードが交絡される) では、2-cycle と非巡回が d分離同値になる。

In [ ]:
ARGS = () if QUICK else ()
if RUN["dsep"]:
    sys.modules["dsep"].main()

### 3.3 欠測が条件付き独立を壊す条件の監査

In [ ]:
%%module ci_audit
"""欠測と削除方式が, どの条件付き独立を壊すかを網羅的に監査する.

制約ベース手法の入力は条件付き独立の判定だけなので, 手法を走らせる前に
「判定そのものがどこで狂うか」を測れば, どの手法がどこで破綻するかは
アルゴリズムの実装に依らずに言える. 個別手法の比較より前にこれを置く.

やること:
  1. 完全データの母共分散から, 観測変数上の真の条件付き独立を列挙する.
  2. 欠測を入れたデータに削除方式 (listwise / testwise) を当てて偏相関を測る.
  3. 真の独立が従属に見える (偽の辺の元) 件数と, その偏りの大きさ.
     真の従属が独立に見える (辺の欠落の元) 件数と, 減衰の大きさ.

標本誤差と偏りを混ぜないため, 大標本 (既定 n=200000) で1回だけ評価し,
判定には p 値ではなく偏相関の絶対値の閾値を使う.

実行:  uv run python src/ci_audit.py [n]
"""
from __future__ import annotations

import itertools
import json
import os
import sys

import numpy as np

from diagnostics import partial_corr, population_cov
from generator import MissingSpec, Scenario, build_B, generate

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
os.makedirs(OUT, exist_ok=True)

ZERO = 1e-8      # 母集団で独立とみなす閾値
BROKEN = 0.05    # 標本でこれ以上の偏相関が出たら独立が壊れたとみなす


def all_tests(names: list[str]):
    d = len(names)
    for i, j in itertools.combinations(range(d), 2):
        rest = [k for k in range(d) if k not in (i, j)]
        for r in range(len(rest) + 1):
            for cond in itertools.combinations(rest, r):
                yield i, j, cond


def label(names, i, j, cond) -> str:
    c = ",".join(names[k] for k in cond) or "{}"
    return f"{names[i]} _||_ {names[j]} | {c}"


def empirical_pcorr(X: np.ndarray, i: int, j: int, cond: tuple[int, ...],
                    deletion: str) -> tuple[float, int]:
    cols = [i, j, *cond]
    if deletion == "listwise":
        rows = ~np.isnan(X).any(axis=1)
    elif deletion == "testwise":
        rows = ~np.isnan(X[:, cols]).any(axis=1)
    else:
        raise ValueError(deletion)
    D = X[np.ix_(rows, cols)]
    if D.shape[0] < len(cols) + 20:
        return float("nan"), int(D.shape[0])
    S = np.corrcoef(D, rowvar=False)
    if not np.all(np.isfinite(S)):
        return float("nan"), int(D.shape[0])
    return partial_corr(S, 0, 1, tuple(range(2, len(cols)))), int(D.shape[0])


def audit_one(sc: Scenario, n: int) -> dict:
    B, names, _ = build_B(sc)
    idx = {v: i for i, v in enumerate(names)}
    keep = [i for i, v in enumerate(names) if not (sc.latent and v == "L")]
    obs = [names[i] for i in keep]
    S_pop = population_cov(B, sc.noise_scale)[np.ix_(keep, keep)]

    out = generate(sc, n)
    X = out["X_obs"]

    res = {"deletions": {}, "n": n}
    for deletion in ("listwise", "testwise"):
        broken, vanished, worst_break, worst_att = [], [], 0.0, 1.0
        n_ci = n_dep = 0
        for i, j, cond in all_tests(obs):
            pop = abs(partial_corr(S_pop, i, j, cond))
            emp, n_used = empirical_pcorr(X, i, j, cond, deletion)
            if emp != emp:
                continue
            if pop < ZERO:
                n_ci += 1
                if abs(emp) > BROKEN:
                    broken.append({"test": label(obs, i, j, cond),
                                   "empirical_pcorr": round(float(emp), 4),
                                   "n_used": n_used})
                worst_break = max(worst_break, abs(emp))
            else:
                n_dep += 1
                ratio = abs(emp) / pop
                if abs(emp) < BROKEN <= pop:
                    vanished.append({"test": label(obs, i, j, cond),
                                     "population_pcorr": round(float(pop), 4),
                                     "empirical_pcorr": round(float(emp), 4),
                                     "n_used": n_used})
                worst_att = min(worst_att, ratio)
        res["deletions"][deletion] = {
            "n_true_ci": n_ci, "n_true_dep": n_dep,
            "n_broken_ci": len(broken), "n_vanished_dep": len(vanished),
            "max_false_pcorr": round(float(worst_break), 4),
            "min_attenuation_ratio": round(float(worst_att), 4),
            "broken": broken[:6], "vanished": vanished[:6],
        }
    return res


def main() -> None:
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 200000
    cases = []
    for cycle in ("acyclic2", "2-cycle"):
        for confound in ("none", "both"):
            for mech, target, rate, driver in [
                    ("none", "Y", 0.0, None),
                    ("mcar", "Y", 0.6, None), ("mar", "Y", 0.6, "Z"),
                    ("mnar_self", "Y", 0.3, None), ("mnar_self", "Y", 0.6, None),
                    ("mcar", "X1", 0.6, None), ("mar", "X1", 0.6, "Z"),
                    ("mnar_self", "X1", 0.3, None), ("mnar_self", "X1", 0.6, None),
                    # 欠測指標が当該変数以外の親も持つ場合. 親が当該変数と隣接しない
                    # (Z) ときだけ, 検定ごと削除でも条件付き独立が壊れる.
                    ("mnar_mixed", "Y", 0.3, "Z"), ("mnar_mixed", "Y", 0.6, "Z"),
                    ("mnar_mixed", "Y", 0.3, "V"), ("mnar_mixed", "Y", 0.6, "V")]:
                cases.append((cycle, confound, mech, target, rate, driver))

    rows = []
    for cycle, confound, mech, target, rate, driver in cases:
        missing = [] if mech == "none" else [
            MissingSpec(target=target, mechanism=mech, rate=rate, driver=driver)]
        # 欠測指標の親を作れるよう, Y に巡回と独立な親 V を持たせた構造で回す
        sc = Scenario(cycle=cycle, rho=0.6, latent=True, confound=confound,
                      y_parent=True, missing=missing, seed=0)
        r = audit_one(sc, n)
        rows.append({"cycle": cycle, "confound": confound, "mechanism": mech,
                     "target": target, "driver": driver,
                     "where": "巡回内" if target == "X1" else "巡回外",
                     "rate": rate, **r})
        print(f"  {cycle:<9} 交絡={confound:<5} {mech:<11} 駆動={str(driver):<5} "
              f"{target:<3} {rate:.2f} 完了", flush=True)

    with open(os.path.join(OUT, "ci_audit.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=1)

    print(f"\n真の条件付き独立が標本で壊れる件数 (|偏相関| > {BROKEN}, n={n})")
    print(f"{'真の構造':<10}{'交絡':<6}{'機構':<11}{'駆動':<5}{'位置':<7}{'欠測率':>6}"
          f"{'list壊れ':>9}{'test壊れ':>9}{'list最大':>10}{'test最大':>10}")
    for r in rows:
        lw, tw = r["deletions"]["listwise"], r["deletions"]["testwise"]
        print(f"{r['cycle']:<10}{r['confound']:<6}{r['mechanism']:<11}"
              f"{str(r['driver']):<5}{r['where']:<7}"
              f"{r['rate']:>6.2f}{lw['n_broken_ci']:>9}{tw['n_broken_ci']:>9}"
              f"{lw['max_false_pcorr']:>10.3f}{tw['max_false_pcorr']:>10.3f}")

    print("\n壊れた独立の例")
    for r in rows:
        for dele in ("listwise", "testwise"):
            for b in r["deletions"][dele]["broken"][:2]:
                print(f"  {r['cycle']:<9} 交絡={r['confound']:<5} {r['mechanism']:<10} "
                      f"{r['where']} {r['rate']:.2f} [{dele}] {b['test']} "
                      f"偏相関={b['empirical_pcorr']}")


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (20000,) if QUICK else (200000,)
if RUN["ci_audit"]:
    run_main("ci_audit", *ARGS)

### 3.4 欠測による従属の減衰

In [ ]:
%%module attenuation
"""欠測機構と削除方式が, 真の従属をどれだけ弱めるかを測る.

ci_audit.py が「真の独立が壊れるか」を測るのに対し, こちらは
「真の従属がどれだけ薄まるか」を測る. 前者は偽の辺, 後者は辺の欠落に対応する.

実行:  uv run python src/attenuation.py [n]
"""
from __future__ import annotations

import json
import os
import sys

import numpy as np

from ci_audit import audit_one
from generator import MissingSpec, Scenario

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")


def main() -> None:
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 200000
    rows = []
    for cycle in ("acyclic2", "2-cycle"):
        for target in ("Y", "X1"):
            for mech in ("mcar", "mar", "mnar_self"):
                for rate in (0.3, 0.6):
                    sp = [MissingSpec(target, mech, rate,
                                      driver="Z" if mech == "mar" else None)]
                    sc = Scenario(cycle=cycle, rho=0.6, latent=True, confound="both",
                                  y_parent=True, missing=sp, seed=0)
                    r = audit_one(sc, n)
                    lw, tw = r["deletions"]["listwise"], r["deletions"]["testwise"]
                    rows.append({
                        "cycle": cycle, "target": target,
                        "where": "巡回内" if target == "X1" else "巡回外",
                        "mechanism": mech, "rate": rate,
                        "listwise": lw["min_attenuation_ratio"],
                        "testwise": tw["min_attenuation_ratio"],
                        "broken_listwise": lw["n_broken_ci"],
                        "broken_testwise": tw["n_broken_ci"],
                        "vanished_listwise": lw["n_vanished_dep"],
                        "vanished_testwise": tw["n_vanished_dep"],
                    })
                    print(f"  {cycle:<9}{target:<3}{mech:<11}{rate:.2f} 完了", flush=True)
    with open(os.path.join(OUT, "attenuation.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=1)
    print(f"\n書き出し: {os.path.join(OUT, 'attenuation.json')}")
    print(f"{'機構':<11}{'欠測率':>6}{'行削除':>9}{'検定削除':>10}{'壊れた独立(行)':>14}")
    for mech in ("mcar", "mar", "mnar_self"):
        for rate in (0.3, 0.6):
            sel = [r for r in rows if r["mechanism"] == mech and r["rate"] == rate]
            print(f"{mech:<11}{rate:>6.2f}"
                  f"{np.mean([r['listwise'] for r in sel]):>9.3f}"
                  f"{np.mean([r['testwise'] for r in sel]):>10.3f}"
                  f"{np.mean([r['broken_listwise'] for r in sel]):>14.1f}")


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (20000,) if QUICK else (200000,)
if RUN["attenuation"]:
    run_main("attenuation", *ARGS)

### 3.5 巡回の判別に効く検定1本に絞った測定

In [ ]:
%%module cycle_missing_probe
"""欠測が巡回の検出をどこで壊すかを, 判別に使う条件付き独立検定1本に絞って測る.

D6c で, 2-cycle X1 <-> X2 が条件付き独立で区別できるのは
  (i) 巡回ノードに観測される外生の親がある (ここでは Z -> X1)
  (ii) 巡回ノード対が未観測交絡で同時に交絡されていない
の2条件がそろうときだと分かった. そのとき判別に効く検定はただ1本で,

      Z _||_ X2 | X1

である. 非巡回 (X1 -> X2) ではこれが成り立ち, 2-cycle では成り立たない.
つまり制約ベース手法にとって, 巡回の検出はこの検定1本の当否に還元される.

ここでは, 欠測がこの検定をどう壊すかを2つの削除方式で測る.
  listwise : どれか1つでも欠測した行を落とす (PC に欠測データを渡す素朴な方法)
  testwise : その検定に出てくる変数だけを見て欠測行を落とす (MVPC の方式)

当初は次の2つの破綻を予想したが, どちらも起きなかった. 理由も分かったので残しておく.

  (A) 欠測が巡回の外 (Y = X1 と X2 の合流点) にあるとき, 行ごと削除は合流点の子で
      選抜したことになるので, 非巡回でも Z と X2 が X1 のもとで従属して見えるはず.
      → 起きない. Z から X2 への道はすべて X1 を通り, X1 は条件付け集合に入っているので,
        Y の子で選抜しても道は開かない. 合流点での選抜が効くのは, 互いに隣接しない
        2つの親の間だけである. 図1 では X1 と X2 が隣接しているので効く余地が無い.
  (B) 欠測が巡回の中 (X1) にあり自己マスキングなら, 条件付け変数自身の値で行が選ばれるので
      検定ごとの削除でも条件付けが歪むはず.
      → 起きない. 条件付け変数の値で選抜しても, その変数で条件付けている限り
        条件付き分布は変わらない.

大標本で偏相関を直接測った結果 (ci_audit.py) でも, どの機構でも偏りは 0.008 以下だった.
欠測が条件付き独立を壊すのは, 欠測変数が互いに隣接しない親を2つ以上持ち, かつ
行ごと削除を使ったときである (ci_audit.py で V -> Y を足して確認).

なお帰無条件 (非巡回・欠測なし) での棄却率は, 反復1000回で
alpha=0.01 のとき 0.008〜0.011, alpha=0.05 のとき 0.048〜0.055, alpha=0.10 のとき
0.101〜0.106 であり, p 値の平均も 0.49 で一様だった (n=500, 2000, 10000)。
検定そのものの較正は取れている. 反復60回で見えた 0.117 は標本誤差である.

実行:  uv run python src/cycle_missing_probe.py           (既定 200 反復, n=2000)
"""
from __future__ import annotations

import json
import os
import sys

import numpy as np
from scipy import stats

from generator import MissingSpec, Scenario, generate

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
os.makedirs(OUT, exist_ok=True)

ALPHA = 0.05


def fisher_z_pvalue(D: np.ndarray) -> float:
    """D = [x, y, z1, ...] の列から偏相関の Fisher-Z 検定の p 値を返す."""
    n = D.shape[0]
    k = D.shape[1] - 2
    if n < k + 10:
        return float("nan")
    S = np.corrcoef(D, rowvar=False)
    if not np.all(np.isfinite(S)):
        return float("nan")
    try:
        P = np.linalg.pinv(S)
    except np.linalg.LinAlgError:
        return float("nan")
    den = np.sqrt(P[0, 0] * P[1, 1])
    r = float(-P[0, 1] / den) if den > 0 else 0.0
    r = min(max(r, -0.999999), 0.999999)
    z = 0.5 * np.log((1 + r) / (1 - r)) * np.sqrt(n - k - 3)
    return float(2 * (1 - stats.norm.cdf(abs(z))))


def one_trial(cycle: str, mech: str, target: str, rate: float, seed: int,
              n: int) -> dict:
    missing = []
    if mech != "none":
        missing = [MissingSpec(target=target, mechanism=mech, rate=rate,
                               driver="Z" if mech == "mar" else None)]
    # 交絡を外し, 巡回が条件付き独立で見えるようにした設定を使う
    sc = Scenario(cycle=cycle, rho=0.6, latent=True, confound="none",
                  missing=missing, seed=seed)
    out = generate(sc, n)
    names = out["obs_names"]
    idx = {v: i for i, v in enumerate(names)}
    X = out["X_obs"]
    cols = [idx["Z"], idx["X2"], idx["X1"]]

    full = X[:, cols]
    listwise = X[~np.isnan(X).any(axis=1)][:, cols]
    tw = X[:, cols]
    testwise = tw[~np.isnan(tw).any(axis=1)]

    return {
        "p_oracle": fisher_z_pvalue(full[~np.isnan(full).any(axis=1)]),
        "p_listwise": fisher_z_pvalue(listwise),
        "p_testwise": fisher_z_pvalue(testwise),
        "n_listwise": int(listwise.shape[0]),
        "n_testwise": int(testwise.shape[0]),
    }


def main() -> None:
    n_rep = int(sys.argv[1]) if len(sys.argv) > 1 else 200
    n = int(sys.argv[2]) if len(sys.argv) > 2 else 2000

    conditions = []
    for cycle in ("acyclic2", "2-cycle"):
        conditions.append((cycle, "none", "Y", 0.0))
        for target in ("Y", "X1"):
            for mech in ("mcar", "mar", "mnar_self"):
                for rate in (0.15, 0.30, 0.60):
                    conditions.append((cycle, mech, target, rate))

    rows = []
    for cycle, mech, target, rate in conditions:
        res = [one_trial(cycle, mech, target, rate, s, n) for s in range(n_rep)]
        def rej(key):
            ps = [r[key] for r in res if r[key] == r[key]]
            return float(np.mean([p < ALPHA for p in ps])) if ps else float("nan")
        rows.append({
            "cycle": cycle, "mechanism": mech, "target": target,
            "where": "巡回内" if target == "X1" else "巡回外", "rate": rate,
            "reject_listwise": rej("p_listwise"),
            "reject_testwise": rej("p_testwise"),
            "n_listwise": float(np.mean([r["n_listwise"] for r in res])),
            "n_testwise": float(np.mean([r["n_testwise"] for r in res])),
            "reps": n_rep, "n": n,
        })
        print(f"  {cycle:<9} {mech:<10} {target:<3} {rate:.2f} 完了", flush=True)

    with open(os.path.join(OUT, "cycle_missing_probe.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=1)

    print("\n判別に使う検定: Z _||_ X2 | X1")
    print("非巡回では成り立つ (棄却率は偽陽性率, 目標 0.05)")
    print("2-cycle では成り立たない (棄却率は検出力, 高いほどよい)\n")
    print(f"{'真の構造':<10}{'機構':<11}{'位置':<7}{'欠測率':>7}"
          f"{'listwise':>10}{'testwise':>10}{'有効n(list)':>12}{'有効n(test)':>12}")
    for r in rows:
        print(f"{r['cycle']:<10}{r['mechanism']:<11}{r['where']:<7}{r['rate']:>7.2f}"
              f"{r['reject_listwise']:>10.3f}{r['reject_testwise']:>10.3f}"
              f"{r['n_listwise']:>12.0f}{r['n_testwise']:>12.0f}")


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (10, 1000) if QUICK else (200, 2000)
if RUN["cycle_missing_probe"]:
    run_main("cycle_missing_probe", *ARGS)

### 3.6 欠測が巡回の判別を壊す条件

欠測指標が巡回ノードと外生変数 Z の両方を親に持つとき、Z _||_ X2 | X1 が壊れて偽の巡回が出る。

In [ ]:
%%module cycle_identification_break
"""欠測が巡回の判別を壊し, 偽の巡回を出す条件を測る.

diagnostics.py の D6c で, 2-cycle X1 <-> X2 が条件付き独立で区別できるのは
  (i) 巡回ノードに観測される外生の親 Z がある
  (ii) 巡回対が未観測交絡で同時に交絡されていない
の2条件がそろうときだけだと分かった. そのとき判別に効く検定はただ1本で

      Z _||_ X2 | X1

である (非巡回では成り立ち, 2-cycle では成り立たない).

ここでは, この検定が欠測でどう壊れるかを, 欠測をどの変数にどの機構で置くかを
振って測る. 非巡回の条件での理論値は 0 なので, そこから離れた分がそのまま
「偽の巡回」の大きさになる.

結論 (実測): 壊れるのは, 検定に現れる変数 X2 が欠測していて, その欠測指標が
X2 自身と検定のもう一方の変数 Z の両方を親に持つときだけである.
このとき検定ごとの削除 (MVPC の方式) でも壊れる.

実行:  uv run python src/cycle_identification_break.py [n]
"""
from __future__ import annotations

import json
import os
import sys

import numpy as np

from diagnostics import partial_corr
from generator import MissingSpec, Scenario, generate

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
os.makedirs(OUT, exist_ok=True)

CASES = [
    ("なし", "none", None),
    ("X2", "mnar_self", None),
    ("X2", "mnar_mixed", "Z"),
    ("X2", "mar", "Z"),
    ("X2", "mcar", None),
    ("Y", "mnar_self", None),
    ("Y", "mnar_mixed", "Z"),
    ("X1", "mnar_self", None),
    ("X1", "mnar_mixed", "Z"),
]


def measure(cycle: str, target: str, mech: str, driver: str | None, rate: float,
            n: int) -> dict:
    missing = [] if mech == "none" else [
        MissingSpec(target=target, mechanism=mech, rate=rate, driver=driver)]
    sc = Scenario(cycle=cycle, rho=0.6, latent=True, confound="none",
                  y_parent=True, missing=missing, seed=0)
    out = generate(sc, n)
    names = out["obs_names"]
    idx = {v: i for i, v in enumerate(names)}
    X = out["X_obs"]
    cols = [idx["Z"], idx["X2"], idx["X1"]]
    lw = X[~np.isnan(X).any(axis=1)][:, cols]
    sub = X[:, cols]
    tw = sub[~np.isnan(sub).any(axis=1)]

    def pc(D):
        if D.shape[0] < 30:
            return float("nan")
        return partial_corr(np.corrcoef(D, rowvar=False), 0, 1, (2,))

    return {"cycle": cycle, "target": target, "mechanism": mech, "driver": driver,
            "rate": rate, "listwise": pc(lw), "testwise": pc(tw),
            "n_listwise": int(lw.shape[0]), "n_testwise": int(tw.shape[0])}


def main() -> None:
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 400000
    rows = []
    for cycle in ("acyclic2", "2-cycle"):
        for target, mech, driver in CASES:
            for rate in ((0.0,) if mech == "none" else (0.30, 0.60)):
                t = "Y" if target == "なし" else target
                rows.append(measure(cycle, t, mech, driver, rate, n))
    with open(os.path.join(OUT, "cycle_identification_break.json"), "w",
              encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=1)

    print("判別に使う検定 pcorr(Z, X2 | X1)")
    print("非巡回の理論値は 0. ここから離れた分が偽の巡回の大きさになる")
    print(f"{'真の構造':<10}{'欠測変数':<7}{'機構':<12}{'指標の親':<7}{'欠測率':>6}"
          f"{'行削除':>10}{'検定削除':>10}")
    for r in rows:
        parent = "---" if r["mechanism"] == "none" else (
            r["target"] if r["mechanism"] == "mnar_self"
            else (r["driver"] or "---") if r["mechanism"] == "mar"
            else f"{r['target']}+{r['driver']}" if r["mechanism"] == "mnar_mixed"
            else "---")
        print(f"{r['cycle']:<10}{r['target']:<7}{r['mechanism']:<12}{parent:<7}"
              f"{r['rate']:>6.2f}{r['listwise']:>10.4f}{r['testwise']:>10.4f}")


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (40000,) if QUICK else (400000,)
if RUN["cycle_identification_break"]:
    run_main("cycle_identification_break", *ARGS)

### 3.7 有限標本での偽の巡回

ここでの MVPC は causal-learn の移植である。公式 R 実装での値は 4.4 節で測る。

In [ ]:
%%module phantom_cycle
"""偽の巡回が有限標本で実際に出るかを, 手法の出力で確かめる.

cycle_identification_break.py は母集団の偏相関で破綻を示した.
ここでは同じ条件を有限標本で回し, 次の2つを測る.

  (1) 巡回の判別器としての棄却率
      判別に効く検定は Z _||_ X2 | X1 の1本なので, 「この検定を棄却したら巡回あり」と
      判定する検出器を作れる. 非巡回のデータでの棄却率が偽陽性率, すなわち偽の巡回の割合になる.
      削除方式は検定ごと (MVPC の方式) を使う.

  (2) PC が Z と X2 の間に辺を残すか
      PC は CPDAG しか出さないので 2-cycle そのものは出力できないが,
      判別に効く検定が壊れれば骨格に Z-X2 の辺が残る. これが観測できる帰結である.
      真のグラフに Z-X2 の辺は無いので, 残れば偽陽性である.

欠測は X2 (巡回ノード) に置き, 機構を振る. mnar_mixed の駆動変数 Z は
巡回の識別に使う外生変数と同じものである.

実行:  uv run python src/phantom_cycle.py [反復数] [n]
"""
from __future__ import annotations

import json
import os
import sys

import numpy as np
from scipy import stats

from generator import MissingSpec, Scenario, generate

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
os.makedirs(OUT, exist_ok=True)

ALPHA = 0.05
MECHS = [("none", None), ("mcar", None), ("mar", "Z"),
         ("mnar_self", None), ("mnar_mixed", "Z")]
RATES = [0.15, 0.30, 0.45, 0.60]


def fisher_z_p(D: np.ndarray) -> float:
    n, k = D.shape[0], D.shape[1] - 2
    if n < k + 10:
        return float("nan")
    S = np.corrcoef(D, rowvar=False)
    if not np.all(np.isfinite(S)):
        return float("nan")
    P = np.linalg.pinv(S)
    den = np.sqrt(P[0, 0] * P[1, 1])
    r = float(-P[0, 1] / den) if den > 0 else 0.0
    r = min(max(r, -0.999999), 0.999999)
    z = 0.5 * np.log((1 + r) / (1 - r)) * np.sqrt(n - k - 3)
    return float(2 * (1 - stats.norm.cdf(abs(z))))


def trial(cycle: str, mech: str, driver: str | None, rate: float, seed: int,
          n: int, run_pc: bool) -> dict:
    missing = [] if mech == "none" else [
        MissingSpec(target="X2", mechanism=mech, rate=rate, driver=driver)]
    sc = Scenario(cycle=cycle, rho=0.6, latent=True, confound="none",
                  y_parent=True, missing=missing, standardize=True, seed=seed)
    out = generate(sc, n)
    names = out["obs_names"]
    idx = {v: i for i, v in enumerate(names)}
    X = out["X_obs"]

    cols = [idx["Z"], idx["X2"], idx["X1"]]
    sub = X[:, cols]
    tw = sub[~np.isnan(sub).any(axis=1)]
    p = fisher_z_p(tw)

    edge = None
    if run_pc:
        from methods import run_method
        from metrics import Estimate
        est, err = run_method("mvpc", X, names)
        if err is None and isinstance(est, Estimate):
            edge = bool(est.adjacent[idx["Z"], idx["X2"]])
    return {"p": p, "edge_Z_X2": edge}


def main() -> None:
    reps = int(sys.argv[1]) if len(sys.argv) > 1 else 200
    n = int(sys.argv[2]) if len(sys.argv) > 2 else 2000
    pc_reps = min(reps, 50)     # MVPC は重いので回数を絞る

    rows = []
    for cycle in ("acyclic2", "2-cycle"):
        for mech, driver in MECHS:
            for rate in ([0.0] if mech == "none" else RATES):
                res = [trial(cycle, mech, driver, rate, s, n, s < pc_reps)
                       for s in range(reps)]
                ps = np.array([r["p"] for r in res if r["p"] == r["p"]])
                edges = [r["edge_Z_X2"] for r in res if r["edge_Z_X2"] is not None]
                rows.append({
                    "cycle": cycle, "mechanism": mech, "driver": driver, "rate": rate,
                    "reject_rate": float(np.mean(ps < ALPHA)) if len(ps) else float("nan"),
                    "mvpc_edge_rate": float(np.mean(edges)) if edges else float("nan"),
                    "reps": int(len(ps)), "pc_reps": len(edges), "n": n,
                })
                print(f"  {cycle:<9}{mech:<11}{rate:.2f} 完了", flush=True)

    with open(os.path.join(OUT, "phantom_cycle.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=1)

    print(f"\n判別器: Z _||_ X2 | X1 を棄却したら巡回ありと判定 (検定ごと削除, alpha={ALPHA})")
    print("非巡回での棄却率が偽の巡回の割合. 2-cycle での棄却率が検出力")
    print(f"MVPC 辺率: MVPC の骨格に Z-X2 の辺が残る割合 (真のグラフには無い辺)\n")
    print(f"{'真の構造':<10}{'機構':<12}{'欠測率':>6}{'棄却率':>9}{'MVPC辺率':>10}")
    for r in rows:
        print(f"{r['cycle']:<10}{r['mechanism']:<12}{r['rate']:>6.2f}"
              f"{r['reject_rate']:>9.3f}{r['mvpc_edge_rate']:>10.3f}")


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (5, 1000) if QUICK else (200, 2000)
if RUN["phantom_cycle"]:
    run_main("phantom_cycle", *ARGS)

### 3.8 評価4軸での既存手法の比較

9月の時点では向き付けの変換に誤りがあった (2.1 で修正済み)。公式実装を加えて回し直したものは 4.9 節にある。

In [ ]:
ARGS = (1, 500) if QUICK else (10, 2000)
if RUN["run_experiment"]:
    run_main("run_experiment", *ARGS)

### 3.9 図の作成

In [ ]:
%%module make_figures
"""実験結果から図を作る.

実行:  uv run python src/make_figures.py
"""
from __future__ import annotations

import csv
import json
import os
import sys
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.family"] = ["Hiragino Sans", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

HERE = os.path.dirname(__file__)
OUT = os.path.join(HERE, "..", "outputs")
FIG = os.path.join(HERE, "..", "figures")
os.makedirs(FIG, exist_ok=True)

SHOW = ["pc_cc", "mvpc", "fci_cc", "direct_lingam", "ica_lingam", "lingd",
        "mlingam", "corr_thresh"]
LABEL = {"pc_cc": "PC (行削除)", "mvpc": "MVPC", "fci_cc": "FCI (行削除)",
         "direct_lingam": "DirectLiNGAM", "ica_lingam": "ICA-LiNGAM",
         "lingd": "LiNG-D", "mlingam": "m-LiNGAM", "corr_thresh": "相関閾値",
         "rcd": "RCD", "pc_impute": "PC (平均代入)", "complete_graph": "全結合"}


def load():
    with open(os.path.join(OUT, "experiment.csv"), encoding="utf-8") as f:
        return list(csv.DictReader(f))


def mean_by(rows, axis, field):
    """(水準 -> 手法 -> 平均) を返す."""
    acc = defaultdict(lambda: defaultdict(list))
    for r in rows:
        if r["axis"] != axis or r["error"]:
            continue
        v = r.get(field, "")
        if v in ("", "nan", None):
            continue
        acc[r["level"]][r["method"]].append(float(v))
    return {lv: {m: float(np.mean(vs)) for m, vs in d.items()} for lv, d in acc.items()}


def line_panel(ax, data, levels, title, ylabel, methods=SHOW):
    xs = np.arange(len(levels))
    for m in methods:
        ys = [data.get(lv, {}).get(m, np.nan) for lv in levels]
        if all(y != y for y in ys):
            continue
        ax.plot(xs, ys, marker="o", ms=3.5, lw=1.2, label=LABEL.get(m, m))
    ax.set_xticks(xs)
    ax.set_xticklabels(levels, fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(alpha=0.3, lw=0.5)
    ax.tick_params(labelsize=8)


def fig_axes(rows):
    specs = [("フィードバック利得", ["0.1", "0.3", "0.5", "0.7", "0.9"], "rho"),
             ("欠測率", ["0.05", "0.15", "0.3", "0.45", "0.6"], "欠測率"),
             ("巡回の複雑さ", ["acyclic2", "2-cycle", "acyclic3", "3-cycle"], "巡回"),
             ("ノイズ分布", ["gaussian", "uniform", "laplace"], "誤差分布")]
    fig, axes = plt.subplots(2, 2, figsize=(9.5, 6.4))
    for ax, (axis, levels, xlab) in zip(axes.ravel(), specs):
        line_panel(ax, mean_by(rows, axis, "skeleton_f1"), levels,
                   f"{axis}", "骨格 F1")
        ax.set_xlabel(xlab, fontsize=9)
    axes[0, 0].legend(fontsize=7, ncol=2, loc="lower left")
    fig.suptitle("評価4軸に沿った既存手法の骨格復元 (10 シード平均, n=2000)", fontsize=11)
    fig.tight_layout()
    p = os.path.join(FIG, "fig_01_axes.png")
    fig.savefig(p, dpi=150)
    print("書き出し:", p)


def fig_cycle(rows):
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
    line_panel(axes[0], mean_by(rows, "ノイズ分布", "cycle_recall"),
               ["gaussian", "uniform", "laplace"], "2-cycle の復元率", "復元率")
    axes[0].set_xlabel("誤差分布", fontsize=9)
    axes[0].set_ylim(-0.05, 1.05)
    line_panel(axes[1], mean_by(rows, "ノイズ分布", "orient_accuracy"),
               ["gaussian", "uniform", "laplace"], "向き付けの正解率", "正解率")
    axes[1].set_xlabel("誤差分布", fontsize=9)
    axes[1].set_ylim(-0.05, 1.05)
    # 凡例は左の面の右下に置く (左上だと LiNG-D の線と重なる). 0 の線にかからないよう少し上げる
    axes[0].legend(fontsize=7, ncol=2, loc="lower right", bbox_to_anchor=(1.0, 0.08))
    fig.suptitle("巡回の復元は誤差分布で決まる (2-cycle, rho=0.6, 自己マスキング30%)",
                 fontsize=11)
    fig.tight_layout()
    p = os.path.join(FIG, "fig_02_cycle.png")
    fig.savefig(p, dpi=150)
    print("書き出し:", p)


def fig_attenuation():
    path = os.path.join(OUT, "attenuation.json")
    if not os.path.exists(path):
        print("attenuation.json が無いので図3は作らない")
        return
    rows = json.load(open(path, encoding="utf-8"))
    mechs = ["mcar", "mar", "mnar_self"]
    names = {"mcar": "MCAR", "mar": "MAR", "mnar_self": "自己マスキング"}
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    w = 0.35
    xs = np.arange(len(mechs) * 2)
    lab, lw_v, tw_v = [], [], []
    for mech in mechs:
        for rate in (0.3, 0.6):
            sel = [r for r in rows if r["mechanism"] == mech and r["rate"] == rate]
            lw_v.append(np.mean([r["listwise"] for r in sel]))
            tw_v.append(np.mean([r["testwise"] for r in sel]))
            lab.append(f"{names[mech]}\n{int(rate*100)}%")
    ax.bar(xs - w / 2, lw_v, w, label="行ごと削除")
    ax.bar(xs + w / 2, tw_v, w, label="検定ごと削除")
    ax.axhline(1.0, color="k", lw=0.8, ls="--")
    ax.set_xticks(xs)
    ax.set_xticklabels(lab, fontsize=8)
    ax.set_ylabel("最小の減衰比", fontsize=9)
    ax.set_title("欠測機構と削除方式による従属の減衰 (1 が無害)", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3, lw=0.5)
    fig.tight_layout()
    p = os.path.join(FIG, "fig_03_attenuation.png")
    fig.savefig(p, dpi=150)
    print("書き出し:", p)


if __name__ == "__main__":
    rows = load()
    fig_axes(rows)
    fig_cycle(rows)
    fig_attenuation()

In [ ]:
if RUN["make_figures"]:
    mf = sys.modules["make_figures"]
    if os.path.exists(os.path.join(OUT_DIR, "experiment.csv")):
        rows = mf.load()
        mf.fig_axes(rows)
        mf.fig_cycle(rows)
    if os.path.exists(os.path.join(OUT_DIR, "attenuation.json")):
        mf.fig_attenuation()
    print("図の出力先:", FIG_DIR)

### 3.10 版の記録 (公式リポジトリのコミット, Tetrad と R の版を含む)

In [ ]:
%%module env_report
"""検証環境の構成を記録する. 結果の再現に必要な版を残すため.

実行:  uv run python src/env_report.py
"""
from __future__ import annotations

import json
import os
import platform
import sys


OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
os.makedirs(OUT, exist_ok=True)

PKGS = ["numpy", "scipy", "pandas", "sklearn", "networkx", "matplotlib",
        "causallearn", "lingam", "torch"]


def main() -> None:
    info = {"python": sys.version.split()[0], "platform": platform.platform(),
            "packages": {}}
    dist = {"sklearn": "scikit-learn", "causallearn": "causal-learn"}
    for name in PKGS:
        try:
            m = __import__(name)
            v = getattr(m, "__version__", None)
            if v is None:
                from importlib.metadata import version
                v = version(dist.get(name, name))
            info["packages"][name] = v
        except Exception as exc:  # noqa: BLE001
            info["packages"][name] = f"未導入 ({type(exc).__name__})"
    # 手法の実装元も残す
    info["methods"] = {
        "pc_cc / pc_impute / mvpc / fci_cc": "causal-learn",
        "direct_lingam / ica_lingam / rcd / mlingam": "lingam",
        "lingd": "本リポジトリ src/methods.py の自作実装 (ICA + 並べ替え探索)",
        "corr_thresh / complete_graph": "本リポジトリの自明なベースライン",
    }
    # 2026-10-09 追加: 論文が示す公式実装 (official/ の下にクローン). コミットを残す
    import subprocess
    root = os.path.join(os.path.dirname(__file__), "..", "official")
    repos = {}
    for d in ("MVPC", "MissDAG", "missnodag", "py-tetrad"):
        try:
            h = subprocess.run(["git", "-C", os.path.join(root, d), "log", "-1",
                                "--format=%H %cd", "--date=short"],
                               capture_output=True, text=True).stdout.strip()
            u = subprocess.run(["git", "-C", os.path.join(root, d), "remote", "get-url",
                                "origin"], capture_output=True, text=True).stdout.strip()
            repos[d] = {"url": u, "commit": h}
        except Exception as exc:  # noqa: BLE001
            repos[d] = f"取得失敗 ({exc})"
    info["official_repos"] = repos
    try:
        from official_tetrad import tetrad_version
        info["tetrad"] = tetrad_version()
    except Exception as exc:  # noqa: BLE001
        info["tetrad"] = f"起動失敗 ({exc})"
    r = subprocess.run(["Rscript", "-e", "cat(R.version.string)"], capture_output=True, text=True)
    info["R"] = r.stdout.strip()
    info["methods_official"] = {
        "ccd_tetrad / fci_tetrad / lingd_tetrad": "Tetrad (py-tetrad 同梱の jar, Java 21)",
        "mvpc_r_td / mvpc_r_permc / mvpc_r_drw": "公式 R 実装 TURuibo/MVPC (official/Rlib の pcalg)",
        "missdag": "公式 ErdunGAO/MissDAG の線形ガウス版 (NOTEARS + EM)",
        "missnodag": "公式 muralikgs/missnodag (official/missnodag/.venv で実行)",
        "mvpc_fixed": "causal-learn の MVPC から欠測指標の親の検出の誤りを直したもの",
    }
    p = os.path.join(OUT, "environment.json")
    with open(p, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=1)
    print("書き出し:", p)
    print(json.dumps(info, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
ARGS = () if QUICK else ()
if RUN["env_report"]:
    run_main("env_report")

## 4. 10月の検証: 論文の公式実装で確かめ直す

### 4.1 9月の実装と公式実装の突き合わせ

同じデータを MVPC (causal-learn / 公式 R)、FCI (causal-learn / Tetrad)、LiNG-D (自作 / Tetrad) に渡して骨格を比べる。

In [ ]:
%%module compare_official
"""9月の環境で使った実装と, 論文が示す公式実装の出力を同じデータで突き合わせる.

  MVPC   : causal-learn の pc(mvpc=True)   と  公式 R 実装 (PermC / DRW)
  FCI    : causal-learn の fci              と  Tetrad の Fci
  LiNG-D : 自作 (methods.ling_d)            と  Tetrad の LingD

同じデータを渡し, 骨格が完全に一致した割合と, 真のグラフに対する骨格 F1 の差を測る.
骨格の一致を第一に見るのは, 向きは出力形式 (CPDAG / PAG / 重み行列) の変換に左右されるが,
骨格はどの形式でも同じ意味を持つためである.

実行:  uv run python src/compare_official.py [シード数] [n]
"""
from __future__ import annotations

import json
import os
import sys
import time

import numpy as np

from generator import MissingSpec, Scenario, generate
from methods import run_fci_cc, run_lingd, run_mvpc
from metrics import score_all, true_structure
from official_missing import run_mvpc_r
from official_tetrad import run_fci, run_lingd as run_lingd_tetrad, tetrad_version

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")

PAIRS = {
    "MVPC (PermC)": (run_mvpc, lambda X, n: run_mvpc_r(X, n, method="permc")),
    "MVPC (DRW)": (run_mvpc, lambda X, n: run_mvpc_r(X, n, method="drw")),
    "FCI": (run_fci_cc, run_fci),
    "LiNG-D": (run_lingd, run_lingd_tetrad),
}

# (表示名, 巡回, 雑音, 欠測の指定)
CONDS = [
    ("非巡回・欠測なし", "acyclic2", "gaussian", []),
    ("2-cycle・欠測なし", "2-cycle", "gaussian", []),
    ("2-cycle・Y に自己マスキング30%", "2-cycle", "gaussian",
     [MissingSpec(target="Y", mechanism="mnar_self", rate=0.3)]),
    ("2-cycle・X2 に MAR30%", "2-cycle", "gaussian",
     [MissingSpec(target="X2", mechanism="mar", rate=0.3, driver="Z")]),
    ("非巡回・X2 に自己+Z 30%", "acyclic2", "gaussian",
     [MissingSpec(target="X2", mechanism="mnar_mixed", rate=0.3, driver="Z")]),
    ("2-cycle・一様雑音・Y に自己マスキング30%", "2-cycle", "uniform",
     [MissingSpec(target="Y", mechanism="mnar_self", rate=0.3)]),
]


def skel(est):
    return est.adjacent.copy()


def main():
    n_seeds = int(sys.argv[1]) if len(sys.argv) > 1 else 10
    n = int(sys.argv[2]) if len(sys.argv) > 2 else 2000
    rows = []
    t0 = time.time()
    for label, cyc, noise, miss in CONDS:
        for seed in range(n_seeds):
            sc = Scenario(cycle=cyc, rho=0.6, latent=True, confound="none", noise=noise,
                          missing=miss, standardize=True, n_extra=6, seed=seed)
            o = generate(sc, n)
            X, names = o["X_obs"], o["obs_names"]
            truth = true_structure(o["B_obs"], names)
            for pname, (f_old, f_new) in PAIRS.items():
                if pname == "LiNG-D" and noise == "gaussian":
                    continue  # LiNG-D は非ガウスの条件だけで比べる
                if pname != "LiNG-D" and noise != "gaussian":
                    continue
                e_old, e_new = f_old(X, names), f_new(X, names)
                a, b = skel(e_old), skel(e_new)
                iu = np.triu_indices(len(names), 1)
                diff = int((a[iu] != b[iu]).sum())
                s_old, s_new = score_all(e_old, truth), score_all(e_new, truth)
                rows.append({"cond": label, "pair": pname, "seed": seed,
                             "skeleton_identical": diff == 0, "skeleton_diff_pairs": diff,
                             "f1_old": s_old["skeleton_f1"], "f1_new": s_new["skeleton_f1"],
                             "c2_old": s_old["recovered_2cycles"],
                             "c2_new": s_new["recovered_2cycles"]})
        print(f"{label} ({time.time() - t0:.0f}s)", flush=True)

    summary = []
    print(f"\nTetrad {tetrad_version()}, n={n}, {n_seeds} シード, 補助変数6本, 標準化あり\n")
    print(f"{'条件':40s} {'比較':14s} {'骨格一致':>8s} {'差の対数':>8s} {'F1(旧)':>8s} {'F1(公式)':>8s}")
    for label, *_ in CONDS:
        for pname in PAIRS:
            rs = [r for r in rows if r["cond"] == label and r["pair"] == pname]
            if not rs:
                continue
            s = {"cond": label, "pair": pname, "n": len(rs),
                 "identical_rate": float(np.mean([r["skeleton_identical"] for r in rs])),
                 "mean_diff_pairs": float(np.mean([r["skeleton_diff_pairs"] for r in rs])),
                 "f1_old": float(np.mean([r["f1_old"] for r in rs])),
                 "f1_new": float(np.mean([r["f1_new"] for r in rs])),
                 "c2_old": float(np.mean([r["c2_old"] for r in rs])),
                 "c2_new": float(np.mean([r["c2_new"] for r in rs]))}
            summary.append(s)
            print(f"{label:40s} {pname:14s} {s['identical_rate']:8.2f} {s['mean_diff_pairs']:8.2f} "
                  f"{s['f1_old']:8.3f} {s['f1_new']:8.3f}")
    json.dump({"n": n, "n_seeds": n_seeds, "tetrad": tetrad_version(),
               "summary": summary, "rows": rows},
              open(os.path.join(OUT, "compare_official.json"), "w"), ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (1, 500) if QUICK else (10, 2000)
if RUN["compare_official"]:
    run_main("compare_official", *ARGS)

### 4.2 CCD の神託出力

d分離の神託で CCD を回し、2-cycle と非巡回を見分ける印があるかを見る。
交絡が両方に入ると 2-cycle と非巡回の出力は同じになり、交絡が無いと Z o-o X2 の隣接で見分けられる。

In [ ]:
from generator import Scenario, generate
from official_tetrad import run_oracle

for cyc in ["acyclic2", "2-cycle", "acyclic3", "3-cycle"]:
    for conf in ["both", "none"]:
        o = generate(Scenario(cycle=cyc, rho=0.6, latent=True, confound=conf, seed=1), 50)
        r = run_oracle("ccd", o["B"], o["meta"]["names"], ["L"])
        print(f"{cyc:9s} 交絡={conf:4s}", r["edges"])

### 4.3 骨格の正解をどこに置くか

係数行列 B の骨格と、神託の骨格 (分離できない対を隣接とみなす) の両方を正解にして F1 を比べる。

In [ ]:
%%module eval_oracle
"""評価の正解をどこに置くかの検査.

9月の run_experiment.py は, 推定した骨格を真の係数行列 B の骨格と比べて F1 を出していた.
しかし巡回や未観測交絡があると, B では隣接しない対でも, 観測変数のどの部分集合で
条件付けても d分離できない対が生じる (例: Z -> X1 <-> X2 で, Z と X2).
条件付き独立だけを使う手法は, 無限標本でもその対に辺を残す. それを偽陽性と数えると,
手法が正しく動いているのに F1 が下がり, 「巡回で劣化した」ように見える.

そこで, 神託 (真のグラフの d分離) で FCI を回した PAG の骨格を「到達可能な正解」とし,
各手法の骨格 F1 を (a) B の骨格 と (b) 神託の骨格 の両方に対して出す.
神託の FCI と CCD の骨格は同じになる (どちらも分離集合の有無で隣接を決める).

実行:  uv run python src/eval_oracle.py [シード数] [n]
"""
from __future__ import annotations

import json
import os
import sys

import numpy as np

from generator import MissingSpec, Scenario, generate
from methods import run_fci_cc, run_mvpc, run_pc_cc
from metrics import Estimate, skeleton_scores, true_structure
from official_missing import run_mvpc_r
from official_tetrad import run_ccd, run_fci, run_oracle

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")

METHODS = {"PC (causal-learn)": run_pc_cc, "MVPC (causal-learn)": run_mvpc,
           "MVPC (公式R)": lambda X, n: run_mvpc_r(X, n, method="permc"),
           "FCI (causal-learn)": run_fci_cc, "FCI (Tetrad)": run_fci,
           "CCD (Tetrad)": run_ccd}

CONDS = [("acyclic2", True), ("2-cycle", True), ("acyclic3", True), ("3-cycle", True),
         ("2-cycle", False)]


def oracle_skeleton(o) -> np.ndarray:
    names_all = o["meta"]["names"]
    lat = [v for v in names_all if v not in o["obs_names"]]
    r = run_oracle("fci", o["B"], names_all, lat)
    obs = o["obs_names"]
    A = np.zeros((len(obs), len(obs)), bool)
    for e in r["edges"]:
        toks = e.split()
        a, b = toks[0], toks[2]
        A[obs.index(a), obs.index(b)] = A[obs.index(b), obs.index(a)] = True
    return A


def main():
    n_seeds = int(sys.argv[1]) if len(sys.argv) > 1 else 10
    n = int(sys.argv[2]) if len(sys.argv) > 2 else 2000
    rows = []
    for cyc, lat in CONDS:
        for seed in range(n_seeds):
            sc = Scenario(cycle=cyc, rho=0.6, latent=lat, confound="both" if lat else "none",
                          noise="gaussian", standardize=True, n_extra=6, seed=seed,
                          missing=[MissingSpec(target="Y", mechanism="mnar_self", rate=0.3)])
            o = generate(sc, n)
            X, names = o["X_obs"], o["obs_names"]
            truth = true_structure(o["B_obs"], names)
            A_or = oracle_skeleton(o)
            oracle = Estimate(names, np.zeros_like(A_or), A_or, kind="oracle")
            iu = np.triu_indices(len(names), 1)
            virtual = int((A_or[iu] & ~truth.adjacent[iu]).sum())
            missing_true = int((~A_or[iu] & truth.adjacent[iu]).sum())
            for m, f in METHODS.items():
                e = f(X, names)
                rows.append({"cycle": cyc, "latent": lat, "seed": seed, "method": m,
                             "virtual_pairs": virtual, "oracle_drops_true": missing_true,
                             "f1_B": skeleton_scores(e, truth)["skeleton_f1"],
                             "f1_oracle": skeleton_scores(e, oracle)["skeleton_f1"],
                             "oracle_f1_vs_B": skeleton_scores(oracle, truth)["skeleton_f1"]})
        print(f"{cyc} latent={lat} 完了", flush=True)

    summ = []
    print(f"\nn={n}, {n_seeds} シード, 補助変数6本, Y に自己マスキング30%, 標準化あり")
    print("仮想隣接 = B では隣接しないが d分離できない対の数. 神託F1 = 神託の骨格を B と比べた F1 (到達上限)")
    print(f"{'巡回':10s}{'交絡':>5s}{'仮想隣接':>8s}{'神託F1':>8s}  " +
          "".join(f"{m[:14]:>16s}" for m in METHODS))
    for cyc, lat in CONDS:
        rs = [r for r in rows if r["cycle"] == cyc and r["latent"] == lat]
        line_b, line_o = [], []
        s = {"cycle": cyc, "latent": lat,
             "virtual_pairs": float(np.mean([r["virtual_pairs"] for r in rs])),
             "oracle_f1_vs_B": float(np.mean([r["oracle_f1_vs_B"] for r in rs]))}
        for m in METHODS:
            mr = [r for r in rs if r["method"] == m]
            s[m] = {"f1_B": float(np.mean([r["f1_B"] for r in mr])),
                    "f1_oracle": float(np.mean([r["f1_oracle"] for r in mr]))}
            line_b.append(f"{s[m]['f1_B']:16.3f}")
            line_o.append(f"{s[m]['f1_oracle']:16.3f}")
        summ.append(s)
        print(f"{cyc:10s}{str(lat):>5s}{s['virtual_pairs']:8.1f}{s['oracle_f1_vs_B']:8.3f}  "
              + "".join(line_b) + "   (B 基準)")
        print(f"{'':31s}" + "".join(line_o) + "   (神託基準)")
    json.dump({"summary": summ, "rows": rows}, open(os.path.join(OUT, "eval_oracle.json"), "w"),
              ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (1, 500) if QUICK else (10, 2000)
if RUN["eval_oracle"]:
    run_main("eval_oracle", *ARGS)

### 4.4 偽の巡回を公式実装で測り直す

CCD (Tetrad), MVPC (公式 R), MissDAG, MissNODAG で、偽の巡回の印が出る割合を測る。
引数は 反復数, n, MissDAG と MissNODAG の反復数。QUICK では後者を 0 にして飛ばす。

In [ ]:
%%module phantom_official
"""偽の巡回 (3.5〜3.7 節) を, 論文が示す公式実装の出力で確かめ直す.

9月の phantom_cycle.py は, 巡回の判別に効く検定 Z _||_ X2 | X1 を単独で使う検出器と,
causal-learn の MVPC の骨格に Z-X2 が残るかを測っていた.
巡回を出力できる手法 (CCD・MissNODAG) は動かしていなかった.
ここでは同じ条件で次を測る.

  CCD (Tetrad)      : 出力の PAG に Z-X2 の隣接があるか. d分離の神託で回すと,
                      交絡の無い 2-cycle では Z-X2 が隣接し, 非巡回では隣接しない
                      (official_tetrad.run_oracle で確認済み). よってこれが CCD における
                      「巡回あり」の印である. 行ごと削除 (論文どおり) と検定ごと削除の両方.
  MVPC (公式 R, PermC) : 骨格に Z-X2 が残るか. 9月の値 (causal-learn) との比較用.
  MissDAG (公式)     : DAG しか出さないので, Z-X2 の隣接だけを記録する.
  MissNODAG (公式)   : 巡回を直接出力する. X1 -> X2 と X2 -> X1 の両方が出たら 2-cycle と判定.
                      論文は介入データを前提にしているが, 本研究の設定 (観測データのみ) で回す.

欠測は X2 (巡回ノード) に置き, 機構を振る. 生成の条件は phantom_cycle.py と同じ
(rho=0.6, 交絡なし, Y に外生の親 V あり, 標準化あり).

実行:  uv run python src/phantom_official.py [反復数] [n] [MissNODAG の反復数]
"""
from __future__ import annotations

import json
import os
import sys
import time

import numpy as np

from generator import MissingSpec, Scenario, generate
from official_missing import run_missdag, run_missnodag, run_mvpc_r
from official_tetrad import run_ccd

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")

MECHS = [("none", None), ("mcar", None), ("mar", "Z"), ("mnar_self", None), ("mnar_mixed", "Z")]
RATES = [0.15, 0.30, 0.60]


def trial(cycle, mech, driver, rate, seed, n, heavy):
    missing = [] if mech == "none" else [
        MissingSpec(target="X2", mechanism=mech, rate=rate, driver=driver)]
    sc = Scenario(cycle=cycle, rho=0.6, latent=True, confound="none",
                  y_parent=True, missing=missing, standardize=True, seed=seed)
    o = generate(sc, n)
    X, names = o["X_obs"], o["obs_names"]
    z, x1, x2 = names.index("Z"), names.index("X1"), names.index("X2")
    rec = {}
    for key, f in [("ccd_lw", lambda: run_ccd(X, names, missing="listwise")),
                   ("ccd_tw", lambda: run_ccd(X, names, missing="testwise")),
                   ("mvpc_r", lambda: run_mvpc_r(X, names, method="permc"))]:
        e = f()
        rec[key] = bool(e.adjacent[z, x2])
    if heavy:
        e = run_missdag(X, names)
        rec["missdag"] = bool(e.adjacent[z, x2])
        try:
            e = run_missnodag(X, names, seed=seed)
            rec["missnodag_2cycle"] = bool(e.directed[x1, x2] and e.directed[x2, x1])
            rec["missnodag_zx2"] = bool(e.adjacent[z, x2])
        except RuntimeError:
            rec["missnodag_2cycle"] = None
            rec["missnodag_zx2"] = None
    return rec


KEYS = ["ccd_lw", "ccd_tw", "mvpc_r", "missdag", "missnodag_2cycle", "missnodag_zx2"]


def main():
    reps = int(sys.argv[1]) if len(sys.argv) > 1 else 20
    n = int(sys.argv[2]) if len(sys.argv) > 2 else 2000
    heavy_reps = int(sys.argv[3]) if len(sys.argv) > 3 else 10
    rows = []
    t0 = time.time()
    for cycle in ("acyclic2", "2-cycle"):
        for mech, driver in MECHS:
            for rate in ([0.0] if mech == "none" else RATES):
                res = [trial(cycle, mech, driver, rate, s, n, s < heavy_reps) for s in range(reps)]
                row = {"cycle": cycle, "mechanism": mech, "rate": rate, "n": n}
                for k in KEYS:
                    v = [r[k] for r in res if r.get(k) is not None]
                    row[k] = float(np.mean(v)) if v else float("nan")
                    row[k + "_reps"] = len(v)
                rows.append(row)
                print(f"  {cycle:<9}{mech:<11}{rate:.2f} ({time.time() - t0:.0f}s)", flush=True)
                json.dump(rows, open(os.path.join(OUT, "phantom_official.json"), "w"),
                          ensure_ascii=False, indent=1)

    print("\n値は「巡回ありの印」が出た割合. 非巡回の行では偽の巡回の割合, 2-cycle の行では検出力")
    print("CCD・MVPC・MissDAG は Z-X2 の隣接, MissNODAG(2c) は X1<->X2 の両方向の出力")
    print(f"{'真の構造':<10}{'機構':<12}{'欠測率':>6}{'CCD行削除':>10}{'CCD検定削除':>11}"
          f"{'MVPC(R)':>9}{'MissDAG':>9}{'MissNODAG(2c)':>14}{'MissNODAG(Z-X2)':>16}")
    for r in rows:
        print(f"{r['cycle']:<10}{r['mechanism']:<12}{r['rate']:>6.2f}{r['ccd_lw']:>10.2f}"
              f"{r['ccd_tw']:>11.2f}{r['mvpc_r']:>9.2f}{r['missdag']:>9.2f}"
              f"{r['missnodag_2cycle']:>14.2f}{r['missnodag_zx2']:>16.2f}")


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (1, 1000, 0) if QUICK else (30, 2000, 10)
if RUN["phantom_official"]:
    run_main("phantom_official", *ARGS)

### 4.5 MVPC の食い違いは乱数の揺れか

公式 R を乱数シードだけ変えて回したときの自己不一致と、causal-learn との不一致を比べる。

In [ ]:
%%module mvpc_seed_check
"""MVPC の causal-learn 版と公式 R 版の食い違いが, PermC の乱数による揺れの範囲かを確かめる.

PermC は欠測指標の親を行ごとに置換するので, 同じデータでも乱数で結果が変わりうる.
公式 R 版を乱数シードだけ変えて回したときの骨格の不一致率 (自己不一致) と,
causal-learn 版と公式 R 版の不一致率を同じデータで比べる.
DRW は乱数を使わないので, R 版どうしは一致するはずである (対照).

実行:  uv run python src/mvpc_seed_check.py [データ数] [R のシード数]
"""
import json
import os
import sys

import numpy as np

from generator import MissingSpec, Scenario, generate
from methods import run_mvpc
from official_missing import run_mvpc_r

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
CONDS = [("2-cycle・Y に自己マスキング30%", "2-cycle", MissingSpec(target="Y", mechanism="mnar_self", rate=0.3)),
         ("2-cycle・X2 に MAR30%", "2-cycle", MissingSpec(target="X2", mechanism="mar", rate=0.3, driver="Z")),
         ("非巡回・X2 に自己+Z 30%", "acyclic2", MissingSpec(target="X2", mechanism="mnar_mixed", rate=0.3, driver="Z"))]


def sk(e):
    iu = np.triu_indices(e.adjacent.shape[0], 1)
    return e.adjacent[iu]


def main():
    n_data = int(sys.argv[1]) if len(sys.argv) > 1 else 10
    n_rseed = int(sys.argv[2]) if len(sys.argv) > 2 else 5
    out = []
    print(f"{'条件':34s}{'補正':>6s}{'R自己不一致':>12s}{'cl対R不一致':>12s}")
    for label, cyc, spec in CONDS:
        for corr in ("permc", "drw"):
            self_dis, cross_dis = [], []
            for d in range(n_data):
                o = generate(Scenario(cycle=cyc, rho=0.6, latent=True, confound="none",
                                      missing=[spec], standardize=True, n_extra=6, seed=d), 2000)
                X, names = o["X_obs"], o["obs_names"]
                rs = [sk(run_mvpc_r(X, names, method=corr, seed=s)) for s in range(n_rseed)]
                cl = sk(run_mvpc(X, names))
                # R のシード対ごとの不一致, causal-learn と各 R の不一致 (骨格が完全一致しない割合)
                self_dis += [float((rs[a] != rs[b]).any()) for a in range(n_rseed) for b in range(a + 1, n_rseed)]
                cross_dis += [float((cl != r).any()) for r in rs]
            row = {"cond": label, "corr": corr, "r_self_disagree": float(np.mean(self_dis)),
                   "cl_vs_r_disagree": float(np.mean(cross_dis))}
            out.append(row)
            print(f"{label:34s}{corr:>6s}{row['r_self_disagree']:12.2f}{row['cl_vs_r_disagree']:12.2f}", flush=True)
    json.dump(out, open(os.path.join(OUT, "mvpc_seed_check.json"), "w"), ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (1, 2) if QUICK else (10, 5)
if RUN["mvpc_seed_check"]:
    run_main("mvpc_seed_check", *ARGS)

### 4.6 causal-learn の MVPC の親の検出の誤りの影響

In [ ]:
%%module mvpc_bug_effect
"""causal-learn の MVPC にある欠測指標の親の検出の誤りが, 結果をどれだけ変えるか.

3つを同じデータで回す.
  cl      : causal-learn 0.1.4.8 の pc(mvpc=True) そのまま (9月の環境が使ったもの)
  cl_fix  : 親の検出だけを直したもの (methods.run_mvpc_fixed)
  R       : 公式 R 実装 (PermC). 乱数による揺れを見るためシードを3つ回す

測るもの
  (1) 欠測指標の親として Z を検出した割合 (mnar_mixed の真の親は X2 自身と Z)
  (2) 公式 R と骨格が一致しない割合
  (3) 偽の巡回の印 (骨格の Z-X2) が出る割合. 非巡回では偽陽性, 2-cycle では検出力

条件は phantom_cycle.py と同じ (rho=0.6, 交絡なし, Y に外生の親 V, 標準化あり, 欠測は X2).

実行:  uv run python src/mvpc_bug_effect.py [反復数] [n]
"""
import json
import os
import sys

import numpy as np

from generator import MissingSpec, Scenario, generate
from methods import _fixed_parent_missingness_pairs, run_mvpc, run_mvpc_fixed
from official_missing import run_mvpc_r

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
CONDS = [("acyclic2", "mnar_self", None, 0.3), ("acyclic2", "mar", "Z", 0.3),
         ("acyclic2", "mnar_mixed", "Z", 0.15), ("acyclic2", "mnar_mixed", "Z", 0.3),
         ("acyclic2", "mnar_mixed", "Z", 0.6),
         ("2-cycle", "mnar_self", None, 0.3), ("2-cycle", "mnar_mixed", "Z", 0.3),
         ("2-cycle", "mnar_mixed", "Z", 0.6)]


def iu(A):
    return A[np.triu_indices(A.shape[0], 1)]


def main():
    reps = int(sys.argv[1]) if len(sys.argv) > 1 else 20
    n = int(sys.argv[2]) if len(sys.argv) > 2 else 2000
    from causallearn.search.ConstraintBased.PC import get_parent_missingness_pairs
    from causallearn.utils.cit import CIT
    rows = []
    print(f"{'構造':9s}{'機構':11s}{'率':>5s} | {'Z検出 cl':>8s}{'cl_fix':>7s} | "
          f"{'R不一致 cl':>10s}{'cl_fix':>7s}{'R自己':>6s} | {'Z-X2 cl':>8s}{'cl_fix':>7s}{'R':>6s}")
    for cyc, mech, drv, rate in CONDS:
        acc = {k: [] for k in ["zdet_cl", "zdet_fix", "dis_cl", "dis_fix", "dis_rr",
                               "zx2_cl", "zx2_fix", "zx2_r"]}
        for s in range(reps):
            o = generate(Scenario(cycle=cyc, rho=0.6, latent=True, confound="none", y_parent=True,
                                  missing=[MissingSpec(target="X2", mechanism=mech, rate=rate,
                                                       driver=drv)],
                                  standardize=True, seed=s), n)
            X, names = o["X_obs"], o["obs_names"]
            z, x2 = names.index("Z"), names.index("X2")
            p_cl = get_parent_missingness_pairs(X, 0.05, CIT(X, "mv_fisherz"), True)
            p_fx = _fixed_parent_missingness_pairs(X, 0.05, CIT(X, "mv_fisherz"), True)
            has_z = lambda p: any(z in list(pr) for m, pr in zip(p["m"], p["prt"]) if m == x2)
            acc["zdet_cl"].append(has_z(p_cl))
            acc["zdet_fix"].append(has_z(p_fx))
            e_cl, e_fx = run_mvpc(X, names), run_mvpc_fixed(X, names)
            e_rs = [run_mvpc_r(X, names, method="permc", seed=k) for k in range(3)]
            rs = [iu(e.adjacent) for e in e_rs]
            acc["dis_cl"].append(np.mean([(iu(e_cl.adjacent) != r).any() for r in rs]))
            acc["dis_fix"].append(np.mean([(iu(e_fx.adjacent) != r).any() for r in rs]))
            acc["dis_rr"].append(np.mean([(rs[a] != rs[b]).any() for a in range(3) for b in range(a + 1, 3)]))
            acc["zx2_cl"].append(e_cl.adjacent[z, x2])
            acc["zx2_fix"].append(e_fx.adjacent[z, x2])
            acc["zx2_r"].append(np.mean([e.adjacent[z, x2] for e in e_rs]))
        row = {"cycle": cyc, "mechanism": mech, "rate": rate, "reps": reps,
               **{k: float(np.mean(v)) for k, v in acc.items()}}
        rows.append(row)
        print(f"{cyc:9s}{mech:11s}{rate:5.2f} | {row['zdet_cl']:8.2f}{row['zdet_fix']:7.2f} | "
              f"{row['dis_cl']:10.2f}{row['dis_fix']:7.2f}{row['dis_rr']:6.2f} | "
              f"{row['zx2_cl']:8.2f}{row['zx2_fix']:7.2f}{row['zx2_r']:6.2f}", flush=True)
    json.dump(rows, open(os.path.join(OUT, "mvpc_bug_effect.json"), "w"), ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (1, 1000) if QUICK else (20, 2000)
if RUN["mvpc_bug_effect"]:
    run_main("mvpc_bug_effect", *ARGS)

### 4.7 MVPC の移植と公式の差を切り分ける

(A) 補正の判定, (B) 補正変数から検定の変数を除くか, (C) 回帰の切片 を1つずつ公式に合わせる。

In [ ]:
%%module mvpc_port_diff
"""causal-learn の MVPC と公式 R 実装の, 補正の段の違いを1つずつ切り分ける.

親の検出の誤りを直しても (methods.run_mvpc_fixed), 公式 R と骨格が食い違う.
補正の段のコードを読み比べると, 違いが2つある.

  (A) 補正をかけるかの判定 (cond_perm_c / cond.PermC)
      R          : 検定の変数に親をもつ欠測指標があり, かつ x と y が骨格上で共通の隣接点をもつ
      causal-learn: 上に加えて, その共通の隣接点が欠測指標の親であることを要求する (より厳しい)
  (B) 補正に使う変数 W から検定の変数を除くか
      R          : 除かない (gaussCItest.permc は ind_W をそのまま使う)
      causal-learn: W_indx = set(W_indx) - set(var) で除き, 空なら補正せずに検定ごと削除に戻る

  (C) 補正の回帰の切片
      R          : lm(x ~ 0 + W) で切片なし
      causal-learn: LinearRegression() で切片あり

研究の中心の検定 Z _||_ X2 | X1 では R_X2 の親が Z なので, (B) により causal-learn は
補正をかけずに素の検定ごと削除に戻る. ここを直すと公式 R に近づくかを測る.

修正の組み合わせ4通り (fix=親の検出のみ, +A, +B, +A+B) を公式 R (PermC, シード3つ) と比べる.

実行:  uv run python src/mvpc_port_diff.py [反復数] [n]
"""
import inspect
import json
import os
import sys
import textwrap

import numpy as np

from generator import MissingSpec, Scenario, generate
from methods import _fixed_parent_missingness_pairs
from metrics import from_causal_learn
from official_missing import run_mvpc_r

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")


def _r_style_cond(X, Y, condition_set, prt_m, skel):
    """R の cond.PermC と同じ判定 (共通の隣接点があればよい)."""
    import networkx as nx
    var = list((X, Y) + condition_set)
    if not (set(var) & set(prt_m["m"])):
        return False
    A = nx.to_numpy_array(skel).astype(int)
    return bool(((A[X, :] == 1) & (A[Y, :] == 1)).sum() > 0)


def _patched_call():
    """MC_FisherZ.__call__ から, W から検定の変数を除く1行だけを外した版を作る."""
    from causallearn.utils import cit
    src = textwrap.dedent(inspect.getsource(cit.MC_FisherZ.__call__))
    old = "W_indx = list(set(W_indx) - set(var))"
    assert old in src
    src = src.replace(old, "W_indx = list(W_indx)")
    ns = {}
    exec(src, cit.__dict__, ns)
    return ns["__call__"]


def _no_intercept_regression(tdel_data, num_model):
    """R と同じく切片なしで回帰する版の learn_regression_model."""
    from sklearn.linear_model import LinearRegression
    ms, rs = [], []
    for i in range(num_model):
        y, W = tdel_data[:, i], tdel_data[:, num_model:]
        m = LinearRegression(fit_intercept=False).fit(W, y)
        ms.append(m)
        rs.append(y - m.predict(W))
    return ms, rs


def run_variant(X, names, A=False, B=False, C=False, alpha=0.05):
    from causallearn.search.ConstraintBased import PC as clpc
    from causallearn.utils import cit
    from causallearn.utils.PCUtils import Helper
    saved = (clpc.get_parent_missingness_pairs, Helper.cond_perm_c, cit.MC_FisherZ.__call__,
             Helper.learn_regression_model)
    clpc.get_parent_missingness_pairs = _fixed_parent_missingness_pairs
    if A:
        Helper.cond_perm_c = _r_style_cond
    if B:
        cit.MC_FisherZ.__call__ = _patched_call()
    if C:
        Helper.learn_regression_model = _no_intercept_regression
    try:
        cg = clpc.pc(X, alpha=alpha, mvpc=True, correction_name="MV_Crtn_Fisher_Z",
                     show_progress=False, node_names=names)
    finally:
        (clpc.get_parent_missingness_pairs, Helper.cond_perm_c, cit.MC_FisherZ.__call__,
         Helper.learn_regression_model) = saved
    return from_causal_learn(cg.G.graph, names, "cpdag")


CONDS = [("acyclic2", "mnar_mixed", 0.15), ("acyclic2", "mnar_mixed", 0.60),
         ("2-cycle", "mnar_mixed", 0.60), ("2-cycle", "mnar_self", 0.30)]
VARIANTS = {"fix": (False, False, False), "fix+A+B": (True, True, False),
            "fix+C": (False, False, True), "fix+A+B+C": (True, True, True)}


def iu(M):
    return M[np.triu_indices(M.shape[0], 1)]


def main():
    reps = int(sys.argv[1]) if len(sys.argv) > 1 else 20
    n = int(sys.argv[2]) if len(sys.argv) > 2 else 2000
    rows = []
    print(f"{'構造':9s}{'機構':11s}{'率':>5s} | 公式Rとの骨格不一致 (R どうし) | Z-X2 が残る割合 (公式R)")
    for cyc, mech, rate in CONDS:
        dis = {k: [] for k in VARIANTS}
        zx2 = {k: [] for k in VARIANTS}
        rr, zr = [], []
        for s in range(reps):
            o = generate(Scenario(cycle=cyc, rho=0.6, latent=True, confound="none", y_parent=True,
                                  missing=[MissingSpec(target="X2", mechanism=mech, rate=rate,
                                                       driver="Z" if mech == "mnar_mixed" else None)],
                                  standardize=True, seed=s), n)
            X, names = o["X_obs"], o["obs_names"]
            z, x2 = names.index("Z"), names.index("X2")
            es = [run_mvpc_r(X, names, method="permc", seed=k) for k in range(3)]
            rs = [iu(e.adjacent) for e in es]
            rr.append(np.mean([(rs[a] != rs[b]).any() for a in range(3) for b in range(a + 1, 3)]))
            zr.append(np.mean([e.adjacent[z, x2] for e in es]))
            for k, (a, b, c) in VARIANTS.items():
                np.random.seed(s)
                e = run_variant(X, names, a, b, c)
                dis[k].append(np.mean([(iu(e.adjacent) != r).any() for r in rs]))
                zx2[k].append(e.adjacent[z, x2])
        row = {"cycle": cyc, "mechanism": mech, "rate": rate, "reps": reps,
               "r_self": float(np.mean(rr)), "r_zx2": float(np.mean(zr)),
               **{f"dis_{k}": float(np.mean(v)) for k, v in dis.items()},
               **{f"zx2_{k}": float(np.mean(v)) for k, v in zx2.items()}}
        rows.append(row)
        print(f"{cyc:9s}{mech:11s}{rate:5.2f} | " +
              " ".join(f"{k}:{row['dis_' + k]:.2f}" for k in VARIANTS) + f" (R:{row['r_self']:.2f}) | " +
              " ".join(f"{k}:{row['zx2_' + k]:.2f}" for k in VARIANTS) + f" (R:{row['r_zx2']:.2f})",
              flush=True)
    json.dump(rows, open(os.path.join(OUT, "mvpc_port_diff.json"), "w"), ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (1, 1000) if QUICK else (20, 2000)
if RUN["mvpc_port_diff"]:
    run_main("mvpc_port_diff", *ARGS)

### 4.8 偽の巡回の割合が欠測率に対して単調でない理由

n は 400000 に固定されている (1分程度)。

In [ ]:
%%module phantom_nonmonotone
"""偽の巡回の割合が欠測率に対して単調でない理由を調べる.

phantom_official.py で, 非巡回・自己+Z の条件の CCD の Z-X2 の辺の残存率が
欠測率 0.15 / 0.30 / 0.60 で 0.27 / 0.03 / 0.30 と単調でなかった.
判別に効く検定 Z _||_ X2 | X1 の偏りは欠測率とともに単調に増える (3.6 節) ので,
PC 型の探索が別の条件付け集合で Z と X2 を分離しているはずである.
大標本 (n=400000, 検定ごと削除) で Z と X2 の偏相関を全条件付け集合について測り,
どの集合が欠測率によって 0 を横切るかを見る.

実行:  uv run python src/phantom_nonmonotone.py
"""
import itertools
import json
import os
import sys

import numpy as np

from generator import MissingSpec, Scenario, generate

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
RATES = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.45, 0.60]


def pcor(D, i, j, S):
    cols = [i, j, *S]
    sub = D[:, cols]
    sub = sub[~np.isnan(sub).any(axis=1)]
    P = np.linalg.inv(np.cov(sub, rowvar=False))
    return float(-P[0, 1] / np.sqrt(P[0, 0] * P[1, 1])), len(sub)


def main():
    n = 400000
    res = {}
    names = None
    for rate in RATES:
        miss = [] if rate == 0 else [MissingSpec(target="X2", mechanism="mnar_mixed", rate=rate, driver="Z")]
        o = generate(Scenario(cycle="acyclic2", rho=0.6, latent=True, confound="none", y_parent=True,
                              missing=miss, standardize=True, seed=0), n)
        X, names = o["X_obs"], o["obs_names"]
        z, x2 = names.index("Z"), names.index("X2")
        others = [k for k in range(len(names)) if k not in (z, x2)]
        row = {}
        for r in range(len(others) + 1):
            for S in itertools.combinations(others, r):
                pc_, m = pcor(X, z, x2, S)
                row["{" + ",".join(names[k] for k in S) + "}"] = (pc_, m)
        res[rate] = row
    keys = list(res[RATES[0]].keys())
    print("Z と X2 の偏相関 (検定ごと削除, n=400000). 列は欠測率. 有限標本 n=2000 で棄却されない目安は |r| < 0.044")
    print(f"{'条件付け集合':16s}" + "".join(f"{r:>8.2f}" for r in RATES))
    for k in keys:
        print(f"{k:16s}" + "".join(f"{res[r][k][0]:8.3f}" for r in RATES))
    json.dump({str(r): {k: v[0] for k, v in res[r].items()} for r in RATES},
              open(os.path.join(OUT, "phantom_nonmonotone.json"), "w"), ensure_ascii=False, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
ARGS = () if QUICK else ()
if RUN["phantom_nonmonotone"]:
    run_main("phantom_nonmonotone")

### 4.9 4軸の実験を公式実装を加えて回し直す

QUICK = False の設定は 5 シード (約26時間)。Tetrad の LiNG-D は13変数・ガウス誤差で終わらないので入れていない。
QUICK では手法を絞る。

In [ ]:
ARGS = (1, 500, '--official', '--only=pc_cc,ccd_tetrad,fci_tetrad,mvpc_r_permc,corr_thresh,complete_graph') if QUICK else (5, 2000, '--official')
if RUN["run_experiment_official"]:
    run_main("run_experiment", *ARGS)

### 4.10 causal-learn の手法だけを向きの修正後に回し直す

結果は experiment_only.csv に書き出し、次のセルで向き付けを集計する。

In [ ]:
ARGS = (1, 500, '--only=pc_cc,mvpc,mvpc_fixed,fci_cc,corr_thresh,complete_graph') if QUICK else (10, 2000, '--only=pc_cc,pc_impute,mvpc,mvpc_fixed,fci_cc,corr_thresh,complete_graph')
if RUN["run_experiment_clfix"]:
    run_main("run_experiment", *ARGS)

In [ ]:
# 向き付けの集計 (決めた割合 / 決めたなかでの正解率). 4.9 と 4.10 で回した結果を読む
import csv, statistics as st

def orient_table(path, methods):
    if not os.path.exists(path):
        print("(無い)", path); return
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    for nz in ["gaussian", "uniform", "laplace"]:
        out = [f"{nz:9s}"]
        for m in methods:
            v = [r for r in rows if r["axis"] == "ノイズ分布" and r["level"] == nz and r["method"] == m
                 and r["error"] == "" and r["orient_accuracy_decided"] not in ("", "nan")]
            if v:
                out.append(f"{m}:{st.mean(float(r['orient_decided_rate']) for r in v):.2f}/"
                           f"{st.mean(float(r['orient_accuracy_decided']) for r in v):.2f}")
        print(" ".join(out))

# run_experiment は --only を付けると名前に _only を足す (4.10 は常に, 4.9 は QUICK のとき)
orient_table(os.path.join(OUT_DIR, "experiment_only.csv"), ["pc_cc", "mvpc", "fci_cc"])
orient_table(os.path.join(OUT_DIR, "experiment_official_only.csv" if QUICK else "experiment_official.csv"),
             ["ccd_tetrad", "fci_tetrad", "mvpc_r_permc", "missdag", "direct_lingam", "lingd"])

### 4.11 MissNODAG を評価軸の要所で回す

1回の学習に 10秒から16分かかる。MissNODAG は観測データだけを渡しているので、論文の想定 (介入データ) の外である。

In [ ]:
%%module missnodag_axes
"""MissNODAG (公式実装) を評価軸の要所だけで回す.

run_experiment.py の全30条件に MissNODAG を入れると1条件あたり約28分かかるので,
巡回を出力できる手法として見るべき条件に絞る. 基準条件は run_experiment.BASE と同じ.

  巡回の複雑さ : acyclic2 / 2-cycle / acyclic3 / 3-cycle   (偽の巡回と巡回の復元)
  欠測機構     : none / mnar_self                          (欠測の影響)
  誤差分布     : uniform                                   (非ガウスで改善するか)

MissNODAG は論文では介入データを前提にしている. ここでは観測データだけを渡すので,
論文の想定の外で使っていることになる. 結果はその前提つきで読む.

実行:  uv run python src/missnodag_axes.py [シード数] [n]
"""
import csv
import os
import sys
import time

import numpy as np

from metrics import score_all, true_structure
from official_missing import run_missnodag
from run_experiment import BASE, make_scenario
from generator import generate

OUT = os.path.join(os.path.dirname(__file__), "..", "outputs")
CONDS = [("巡回", "cycle", v) for v in ["acyclic2", "2-cycle", "acyclic3", "3-cycle"]] + \
        [("欠測機構", "mech", v) for v in ["none"]] + [("誤差分布", "noise", "uniform")]


def main():
    n_seeds = int(sys.argv[1]) if len(sys.argv) > 1 else 3
    n = int(sys.argv[2]) if len(sys.argv) > 2 else 2000
    rows = []
    for axis, key, lv in CONDS:
        cfg = {**BASE, key: lv}
        for seed in range(n_seeds):
            o = generate(make_scenario(cfg, seed), n)
            X, names = o["X_obs"], o["obs_names"]
            truth = true_structure(o["B_obs"], names)
            t0 = time.time()
            try:
                e = run_missnodag(X, names, seed=seed)
                s = score_all(e, truth)
                err = ""
            except RuntimeError as ex:
                s, err = {}, str(ex)[:200]
            rows.append({"axis": axis, "level": lv, "seed": seed, "error": err,
                         "seconds": round(time.time() - t0, 1), **s})
            print(f"{axis}={lv} seed={seed} F1={s.get('skeleton_f1', float('nan')):.3f} "
                  f"2c={s.get('recovered_2cycles')}/{s.get('true_2cycles')} "
                  f"cycle_out={s.get('estimate_has_cycle')} ({time.time() - t0:.0f}s) {err}", flush=True)
    keys = sorted({k for r in rows for k in r})
    with open(os.path.join(OUT, "missnodag_axes.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)


if __name__ == "__main__":
    main()

In [ ]:
ARGS = (1, 500) if QUICK else (3, 2000)
if RUN["missnodag_axes"]:
    run_main("missnodag_axes", *ARGS)